# Machine Learning para MS-CFLP-CI (Multi-Site Capacitated Facility Location Problem with Incompatibility Constraints)

### Descripción del problema

Dado un conjunto de almacenes candidatos y tiendas, hay que decidir:
1. Qué almacenes abrir (pagando un costo fijo por cada uno).
2. A qué almacén asignar cada tienda (pagando costo de suministro x demanda).

Restricciones:
- La suma de demandas asignadas a cada almacén no puede superar su capacidad.
- Algunos pares de tiendas son **incompatibles**: no pueden asignarse al mismo almacén.

El objetivo es minimizar el costo total (costos fijos + costos de suministro).

## Imports

Librerías estándar, `numpy` y `sklearn` (regresión, árboles, random forest).

In [374]:
import io
import contextlib
import random
import re
import sys
import math
import time
from dataclasses import dataclass
from typing import TypedDict

import numpy as np
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.tree import DecisionTreeRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel


### Estructuras de datos

`DatosCFLP` es un `TypedDict` con los campos del problema leidos desde el `.dzn`.
Las soluciones se representan como `(abierta: list[bool], asignada: list[int], restante: list[int])`.
La lista `restante` actua como cache de capacidades residuales.

In [375]:
@dataclass
class ResultadoBenchmark:
    nombre: str
    costo_final: float
    tiempo_total_s: float
    tiempo_entrenamiento_ml_s: float = 0.0
    tiempo_inferencia_ml_s: float = 0.0
    modelo_ml: object = None

    @property
    def tiempo_algoritmo_puro_s(self) -> float:
        return (
            self.tiempo_total_s
            - self.tiempo_entrenamiento_ml_s
            - self.tiempo_inferencia_ml_s
        )

    @property
    def overhead_ml_pct(self) -> float:
        overhead = self.tiempo_entrenamiento_ml_s + self.tiempo_inferencia_ml_s
        return overhead / self.tiempo_total_s * 100 if self.tiempo_total_s else 0.0

In [376]:
class DatosCFLP(TypedDict):
    numero_instalaciones: int
    numero_clientes: int
    capacidad_instalacion: list[int]
    costo_fijo_instalacion: list[int]
    demanda_cliente: list[int]
    costo_envio: list[list[int]]
    pares_incompatibles: list[tuple[int, int]]


SolucionCFLP = tuple[list[bool], list[int], list[int]]

## Lectura de instancia (`.dzn`)

In [377]:
def leer_instancia_dzn(ruta_archivo: str) -> DatosCFLP:
    with open(ruta_archivo, "r") as archivo:
        contenido_completo = archivo.read()

    numero_instalaciones = int(
        extraer_valor_escalar(contenido_completo, "Warehouses")
    )

    numero_clientes = int(
        extraer_valor_escalar(contenido_completo, "Stores")
    )

    capacidad_instalacion = extraer_lista_enteros(
        contenido_completo, "Capacity"
    )

    costo_fijo_instalacion = extraer_lista_enteros(
        contenido_completo, "FixedCost"
    )

    demanda_cliente = extraer_lista_enteros(contenido_completo, "Goods")

    costo_envio_raw = extraer_matriz_enteros(
        contenido_completo, "SupplyCost",
        numero_clientes, numero_instalaciones
    )
    costo_envio = [list(col) for col in zip(*costo_envio_raw)]

    pares_incompatibles = extraer_pares_incompatibles(contenido_completo)

    return {
        "numero_instalaciones": numero_instalaciones,
        "numero_clientes": numero_clientes,
        "capacidad_instalacion": capacidad_instalacion,
        "costo_fijo_instalacion": costo_fijo_instalacion,
        "demanda_cliente": demanda_cliente,
        "costo_envio": costo_envio,
        "pares_incompatibles": pares_incompatibles,
    }



def extraer_valor_escalar(texto: str, nombre_variable: str) -> str:
    patron = rf"{nombre_variable}\s*=\s*([0-9]+)\s*;"
    coincidencia = re.search(patron, texto)
    if not coincidencia:
        raise ValueError(
            f"No se encontró la variable '{nombre_variable}' en el archivo."
        )
    return coincidencia.group(1)



def extraer_lista_enteros(texto: str, nombre_variable: str) -> list[int]:
    patron = rf"{nombre_variable}\s*=\s*\[([^\]]+)\]"
    coincidencia = re.search(patron, texto, re.DOTALL)
    if not coincidencia:
        raise ValueError(
            f"No se encontró la lista '{nombre_variable}' en el archivo."
        )
    valores_texto = coincidencia.group(1)
    return [int(v.strip()) for v in valores_texto.split(",") if v.strip()]



def extraer_matriz_enteros(
    texto: str,
    nombre_variable: str,
    numero_filas: int,
    numero_columnas: int,
) -> list[list[int]]:
    patron = rf"{nombre_variable}\s*=\s*\[([^\]]+)\]"
    coincidencia = re.search(patron, texto, re.DOTALL)
    if not coincidencia:
        raise ValueError(
            f"No se encontró la matriz '{nombre_variable}' en el archivo."
        )

    bloque = coincidencia.group(1)
    filas_texto = bloque.split("|")

    matriz = []
    for fila_texto in filas_texto:
        fila_texto = fila_texto.strip().strip(",").strip()
        if not fila_texto:
            continue
        valores_fila = [
            int(v.strip()) for v in fila_texto.split(",") if v.strip()
        ]
        if valores_fila:
            matriz.append(valores_fila)

    return matriz



def extraer_pares_incompatibles(texto: str) -> list[tuple[int, int]]:
    patron = r"IncompatiblePairs\s*=\s*\[([^\]]+)\]"
    coincidencia = re.search(patron, texto, re.DOTALL)
    if not coincidencia:
        return []
    bloque = coincidencia.group(1)
    pares = []
    for fila in bloque.split("|"):
        fila = fila.strip().strip(",").strip()
        if not fila:
            continue
        partes = [v.strip() for v in fila.split(",") if v.strip()]
        if len(partes) >= 2:
            pares.append((int(partes[0]) - 1, int(partes[1]) - 1))
    return pares



def _precomputar_incompatibles(
    numero_clientes: int,
    pares_incompatibles: list[tuple[int, int]],
) -> list[set[int]]:
    incompatibles: list[set[int]] = [set() for _ in range(numero_clientes)]
    for a, b in pares_incompatibles:
        incompatibles[a].add(b)
        incompatibles[b].add(a)
    return incompatibles

## Función de costo total

In [378]:
def calcular_costo_total(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
) -> int:
    costo_por_abrir_instalaciones = sum(
        costo_fijo_instalacion[i]
        for i, abierta in enumerate(instalacion_esta_abierta)
        if abierta
    )

    costo_por_envios_a_clientes = sum(
        demanda_cliente[j]
        * costo_envio[instalacion_asignada_al_cliente[j]][j]
        for j in range(len(instalacion_asignada_al_cliente))
    )

    return costo_por_abrir_instalaciones + costo_por_envios_a_clientes


## 1. Busqueda local

Cuatro movimientos aparecen una y otra vez a lo largo del cuaderno: reasignar
un cliente (N1), cerrar una instalacion (N2), abrir una instalacion (N3) e
intercambiar dos clientes entre instalaciones (N4). GRASP, GVNS y SA los usan
todos, cada uno con su propia politica de aceptacion (barrido completo,
primera mejora, o Metropolis), pero el calculo del delta de coste de cada
movimiento es exactamente el mismo calculo en los cuatro casos. Se define una
sola vez aqui, y cada algoritmo posterior solo aporta su politica de cuando
aplicarlo.

Estas funciones **no aplican el movimiento ni deciden si conviene**: solo
calculan el delta de coste (y, en N2/N3/N4, que reasignacion concreta lo
produce). Quien llama decide -- GRASP/GVNS aplican solo si el delta es
negativo (rentable); SA le pasa el delta, sea cual sea su signo, al criterio
de Metropolis.

### N1: Reasignar un cliente

Reasignar un cliente de su instalación actual a otra factible. El delta de
coste tiene tres componentes: la diferencia de coste de envío entre origen y
destino, el coste fijo de abrir la instalación destino si estaba cerrada, y
el ahorro de coste fijo si el cliente era el único que quedaba en la
instalación de origen (con lo que esta cierra).

In [379]:
def _delta_reasignar_cliente(
    indice_cliente: int,
    instalacion_actual: int,
    instalacion_destino: int,
    instalacion_esta_abierta: list[bool],
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    es_el_unico_en_origen: bool,
) -> int:
    demanda = demanda_cliente[indice_cliente]
    costo_envio_actual = costo_envio[instalacion_actual][indice_cliente] * demanda
    costo_envio_nuevo  = costo_envio[instalacion_destino][indice_cliente] * demanda
    ahorro_cierre_origen = (
        costo_fijo_instalacion[instalacion_actual] if es_el_unico_en_origen else 0
    )
    coste_apertura_destino = (
        costo_fijo_instalacion[instalacion_destino]
        if not instalacion_esta_abierta[instalacion_destino] else 0
    )
    return costo_envio_nuevo + coste_apertura_destino - costo_envio_actual - ahorro_cierre_origen

def _reasignar_clientes_en_barrido(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]] | None = None,
) -> tuple[SolucionCFLP, bool]:
    instalacion_esta_abierta = list(instalacion_esta_abierta)
    instalacion_asignada_al_cliente = list(instalacion_asignada_al_cliente)
    capacidad_restante_instalacion = list(capacidad_restante_instalacion)

    hubo_mejora = False

    for indice_cliente in range(numero_clientes):
        instalacion_actual = instalacion_asignada_al_cliente[indice_cliente]
        demanda = demanda_cliente[indice_cliente]
        inc = incompatibles_cliente[indice_cliente] if incompatibles_cliente else set()
        prohibidas = {instalacion_asignada_al_cliente[k] for k in inc}

        es_el_unico = sum(
            1 for j in range(numero_clientes)
            if instalacion_asignada_al_cliente[j] == instalacion_actual
        ) == 1

        for instalacion_destino in range(numero_instalaciones):
            if instalacion_destino == instalacion_actual:
                continue
            if capacidad_restante_instalacion[instalacion_destino] < demanda:
                continue
            if instalacion_destino in prohibidas:
                continue

            variacion = _delta_reasignar_cliente(
                indice_cliente, instalacion_actual, instalacion_destino,
                instalacion_esta_abierta, costo_fijo_instalacion, demanda_cliente,
                costo_envio, es_el_unico,
            )

            if variacion < 0:
                capacidad_restante_instalacion[instalacion_actual] += demanda
                if es_el_unico:
                    instalacion_esta_abierta[instalacion_actual] = False
                instalacion_asignada_al_cliente[indice_cliente] = instalacion_destino
                capacidad_restante_instalacion[instalacion_destino] -= demanda
                instalacion_esta_abierta[instalacion_destino] = True

                instalacion_actual = instalacion_destino
                es_el_unico = False
                hubo_mejora = True

    return (
        instalacion_esta_abierta,
        instalacion_asignada_al_cliente,
        capacidad_restante_instalacion,
    ), hubo_mejora

### N2 Cerrar una instalación

Cerrar una instalación abierta, reasignando a todos sus clientes en la
alternativa factible más barata disponible en ese momento. El delta acumula,
cliente a cliente, el incremento en coste de envío --más el coste fijo de
abrir una alternativa que estuviera cerrada, contado una sola vez aunque
varios clientes se reasignen a ella-- menos el ahorro del coste fijo de la
instalación que se cierra.

In [380]:
def _evaluar_cierre_instalacion(
    instalacion: int,
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]],
) -> tuple[int, dict[int, int]] | None:
    clientes_asignados = [
        j for j in range(numero_clientes)
        if instalacion_asignada_al_cliente[j] == instalacion
    ]
    if not clientes_asignados:
        return None

    ahorro_al_cerrar = costo_fijo_instalacion[instalacion]
    incremento_en_envios = 0
    reasignacion: dict[int, int] = {}
    demanda_comprometida: dict[int, int] = {}
    instalaciones_recien_abiertas: set[int] = set()

    for cliente in clientes_asignados:
        d = demanda_cliente[cliente]
        costo_actual = costo_envio[instalacion][cliente] * d
        inc = incompatibles_cliente[cliente]
        prohibidas = {instalacion_asignada_al_cliente[k] for k in inc}
        prohibidas |= {reasignacion[k] for k in inc if k in reasignacion}

        mejor_alt, mejor_costo_alt = None, float("inf")
        for i_alt in range(numero_instalaciones):
            if i_alt == instalacion or i_alt in prohibidas:
                continue
            cap_disponible = (
                capacidad_restante_instalacion[i_alt] - demanda_comprometida.get(i_alt, 0)
            )
            if cap_disponible < d:
                continue
            c = costo_envio[i_alt][cliente] * d
            if not instalacion_esta_abierta[i_alt] and i_alt not in instalaciones_recien_abiertas:
                c += costo_fijo_instalacion[i_alt]
            if c < mejor_costo_alt:
                mejor_costo_alt, mejor_alt = c, i_alt

        if mejor_alt is None:
            return None

        incremento_en_envios += mejor_costo_alt - costo_actual
        reasignacion[cliente] = mejor_alt
        demanda_comprometida[mejor_alt] = demanda_comprometida.get(mejor_alt, 0) + d
        if not instalacion_esta_abierta[mejor_alt]:
            instalaciones_recien_abiertas.add(mejor_alt)

    delta = incremento_en_envios - ahorro_al_cerrar
    return delta, reasignacion

def _cerrar_instalaciones_poco_rentables(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]] | None = None,
) -> tuple[SolucionCFLP, bool]:
    instalacion_esta_abierta = list(instalacion_esta_abierta)
    instalacion_asignada_al_cliente = list(instalacion_asignada_al_cliente)
    capacidad_restante_instalacion = list(capacidad_restante_instalacion)
    incompatibles = incompatibles_cliente or [set() for _ in range(numero_clientes)]

    hubo_mejora = False

    for indice_instalacion_a_cerrar in range(numero_instalaciones):
        if not instalacion_esta_abierta[indice_instalacion_a_cerrar]:
            continue

        propuesta = _evaluar_cierre_instalacion(
            indice_instalacion_a_cerrar,
            instalacion_esta_abierta, instalacion_asignada_al_cliente,
            capacidad_restante_instalacion, numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio, incompatibles,
        )
        if propuesta is None:
            continue
        delta, reasignacion = propuesta

        if delta < 0:
            for ic, nueva_instalacion in reasignacion.items():
                demanda_ic = demanda_cliente[ic]
                capacidad_restante_instalacion[indice_instalacion_a_cerrar] += demanda_ic
                instalacion_asignada_al_cliente[ic] = nueva_instalacion
                capacidad_restante_instalacion[nueva_instalacion] -= demanda_ic
                instalacion_esta_abierta[nueva_instalacion] = True
            instalacion_esta_abierta[indice_instalacion_a_cerrar] = False
            hubo_mejora = True

    return (
        instalacion_esta_abierta,
        instalacion_asignada_al_cliente,
        capacidad_restante_instalacion,
    ), hubo_mejora


### N3: Abrir una instalación cerrada

Abrir una instalación cerrada, migrando a ella los clientes que más ahorran
enviándose desde allí en vez de su instalación actual, hasta agotar su
capacidad. El delta resta al coste fijo de abrir el ahorro de envío
acumulado, y también el coste fijo de cualquier instalación de origen que se
quede sin clientes y por tanto cierre como efecto colateral --sin este
término se subestimaría el beneficio real de abrir.

In [381]:
def _evaluar_apertura_instalacion(
    instalacion: int,
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_clientes: int,
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]],
    carga_por_instalacion: list[int],
) -> tuple[int, list[int]] | None:
    candidatos_ahorro = []
    for cliente in range(numero_clientes):
        actual = instalacion_asignada_al_cliente[cliente]
        d = demanda_cliente[cliente]
        ahorro = costo_envio[actual][cliente] * d - costo_envio[instalacion][cliente] * d
        if ahorro > 0:
            candidatos_ahorro.append((ahorro, cliente))

    if not candidatos_ahorro:
        return None

    candidatos_ahorro.sort(reverse=True)

    seleccionados: list[int] = []
    capacidad_disponible = capacidad_restante_instalacion[instalacion]
    ahorro_acumulado = 0
    excluidos_acumulados: set[int] = set()

    for ahorro, cliente in candidatos_ahorro:
        d = demanda_cliente[cliente]
        if d > capacidad_disponible or cliente in excluidos_acumulados:
            continue
        seleccionados.append(cliente)
        capacidad_disponible -= d
        ahorro_acumulado += ahorro
        excluidos_acumulados |= incompatibles_cliente[cliente]

    if not seleccionados:
        return None

    seleccionados_por_origen: dict[int, int] = {}
    for cliente in seleccionados:
        origen = instalacion_asignada_al_cliente[cliente]
        seleccionados_por_origen[origen] = seleccionados_por_origen.get(origen, 0) + 1

    ahorro_por_cierres_origen = 0
    for origen, n_seleccionados_origen in seleccionados_por_origen.items():
        if n_seleccionados_origen == carga_por_instalacion[origen]:
            ahorro_por_cierres_origen += costo_fijo_instalacion[origen]

    delta = costo_fijo_instalacion[instalacion] - ahorro_acumulado - ahorro_por_cierres_origen
    return delta, seleccionados

def _abrir_instalaciones_rentables(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]] | None = None,
) -> tuple[SolucionCFLP, bool]:
    instalacion_esta_abierta = list(instalacion_esta_abierta)
    instalacion_asignada_al_cliente = list(instalacion_asignada_al_cliente)
    capacidad_restante_instalacion = list(capacidad_restante_instalacion)
    incompatibles = incompatibles_cliente or [set() for _ in range(numero_clientes)]

    carga_por_instalacion = [0] * numero_instalaciones
    for j in range(numero_clientes):
        carga_por_instalacion[instalacion_asignada_al_cliente[j]] += 1

    hubo_mejora = False

    for indice_instalacion_a_abrir in range(numero_instalaciones):
        if instalacion_esta_abierta[indice_instalacion_a_abrir]:
            continue

        propuesta = _evaluar_apertura_instalacion(
            indice_instalacion_a_abrir,
            instalacion_esta_abierta, instalacion_asignada_al_cliente,
            capacidad_restante_instalacion, numero_clientes, costo_fijo_instalacion,
            demanda_cliente, costo_envio, incompatibles, carga_por_instalacion,
        )
        if propuesta is None:
            continue
        delta, seleccionados = propuesta

        if delta >= 0:
            continue

        instalaciones_origen = {
            instalacion_asignada_al_cliente[indice_cliente] for indice_cliente in seleccionados
        }

        for indice_cliente in seleccionados:
            instalacion_origen = instalacion_asignada_al_cliente[indice_cliente]
            demanda = demanda_cliente[indice_cliente]
            capacidad_restante_instalacion[instalacion_origen] += demanda
            capacidad_restante_instalacion[indice_instalacion_a_abrir] -= demanda
            instalacion_asignada_al_cliente[indice_cliente] = indice_instalacion_a_abrir
            carga_por_instalacion[instalacion_origen] -= 1
            carga_por_instalacion[indice_instalacion_a_abrir] += 1
        instalacion_esta_abierta[indice_instalacion_a_abrir] = True

        for instalacion_origen in instalaciones_origen:
            if carga_por_instalacion[instalacion_origen] == 0:
                instalacion_esta_abierta[instalacion_origen] = False

        hubo_mejora = True

    return (
        instalacion_esta_abierta,
        instalacion_asignada_al_cliente,
        capacidad_restante_instalacion,
    ), hubo_mejora


### N4: Intercambio de dos clientes (swap)

Intercambia las asignaciones de dos clientes que estan en instalaciones
distintas, sin abrir ni cerrar ninguna: cada una pierde un ocupante y gana
otro, asi que el delta es puramente de envio, sin componente de coste fijo.
A diferencia de N1/N2/N3, no depende de que exista hueco libre en alguna
instalacion, asi que sigue encontrando mejoras incluso cuando todas las
instalaciones abiertas estan casi llenas -- justo donde N1 se queda sin
movimientos disponibles.

In [382]:
def _delta_intercambiar_clientes(
    cliente_a: int,
    cliente_b: int,
    instalacion_a: int,
    instalacion_b: int,
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
) -> int:
    da = demanda_cliente[cliente_a]
    db = demanda_cliente[cliente_b]
    envio_actual = (
        costo_envio[instalacion_a][cliente_a] * da
        + costo_envio[instalacion_b][cliente_b] * db
    )
    envio_nuevo = (
        costo_envio[instalacion_b][cliente_a] * da
        + costo_envio[instalacion_a][cliente_b] * db
    )
    return envio_nuevo - envio_actual


def _intercambiar_clientes_en_barrido(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]] | None = None,
) -> tuple[SolucionCFLP, bool]:
    instalacion_esta_abierta = list(instalacion_esta_abierta)
    instalacion_asignada_al_cliente = list(instalacion_asignada_al_cliente)
    capacidad_restante_instalacion = list(capacidad_restante_instalacion)
    incompatibles = incompatibles_cliente or [set() for _ in range(numero_clientes)]

    clientes_por_instalacion = [set() for _ in range(numero_instalaciones)]
    for j in range(numero_clientes):
        clientes_por_instalacion[instalacion_asignada_al_cliente[j]].add(j)

    hubo_mejora = False

    for cliente_a in range(numero_clientes):
        inc_a = incompatibles[cliente_a]
        cambio = True
        while cambio:
            cambio = False
            instalacion_a = instalacion_asignada_al_cliente[cliente_a]
            demanda_a = demanda_cliente[cliente_a]
            costo_actual_a = costo_envio[instalacion_a][cliente_a]

            candidato = None

            for instalacion_b in range(numero_instalaciones):
                if instalacion_b == instalacion_a:
                    continue
                if costo_envio[instalacion_b][cliente_a] >= costo_actual_a:
                    continue

                for cliente_b in list(clientes_por_instalacion[instalacion_b]):
                    demanda_b = demanda_cliente[cliente_b]

                    if capacidad_restante_instalacion[instalacion_a] + demanda_a < demanda_b:
                        continue
                    if capacidad_restante_instalacion[instalacion_b] + demanda_b < demanda_a:
                        continue

                    if any(
                        k != cliente_b and instalacion_asignada_al_cliente[k] == instalacion_b
                        for k in inc_a
                    ):
                        continue
                    if any(
                        k != cliente_a and instalacion_asignada_al_cliente[k] == instalacion_a
                        for k in incompatibles[cliente_b]
                    ):
                        continue

                    variacion = _delta_intercambiar_clientes(
                        cliente_a, cliente_b, instalacion_a, instalacion_b,
                        demanda_cliente, costo_envio,
                    )
                    if variacion < 0:
                        candidato = (cliente_b, instalacion_b, variacion)
                        break
                if candidato is not None:
                    break

            if candidato is not None:
                cliente_b, instalacion_b, _ = candidato
                demanda_b = demanda_cliente[cliente_b]
                instalacion_asignada_al_cliente[cliente_a] = instalacion_b
                instalacion_asignada_al_cliente[cliente_b] = instalacion_a
                capacidad_restante_instalacion[instalacion_a] += demanda_a - demanda_b
                capacidad_restante_instalacion[instalacion_b] += demanda_b - demanda_a
                clientes_por_instalacion[instalacion_a].discard(cliente_a)
                clientes_por_instalacion[instalacion_a].add(cliente_b)
                clientes_por_instalacion[instalacion_b].discard(cliente_b)
                clientes_por_instalacion[instalacion_b].add(cliente_a)
                hubo_mejora = True
                cambio = True

    return (
        instalacion_esta_abierta,
        instalacion_asignada_al_cliente,
        capacidad_restante_instalacion,
    ), hubo_mejora

## 2. GRASP

GRASP (Greedy Randomized Adaptive Search Procedure) construye en cada iteración
una solución greedy-aleatoria, controlada por el parámetro alfa, y la mejora con
búsqueda local hasta mínimo local. El bucle se repite N veces guardando la mejor.

### Estructuras de datos (GRASP)

In [383]:
class ResultadoGRASP(TypedDict):
    costo_total_minimo: float
    instalaciones_abiertas: list[bool]
    cliente_asignado_a_instalacion: list[int]
    capacidad_restante_por_instalacion: list[int]

### Fase 1: Construccion greedy aleatoria

`alfa` controla cuantas instalaciones entran en la Lista Restringida de Candidatos (RCL):
con `alfa=0` solo entra la mas barata; con `alfa=1` entran todas las factibles.
El umbral es `costo_min + alfa*(costo_max - costo_min)`.
Los clientes se procesan en orden descendente de demanda.

In [384]:
def construir_solucion_greedy_aleatoria(
    numero_instalaciones: int,
    numero_clientes: int,
    capacidad_instalacion: list[int],
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    alfa: float,
    incompatibles_cliente: list[set[int]] | None = None,
) -> SolucionCFLP:
    instalacion_esta_abierta = [False] * numero_instalaciones
    instalacion_asignada_al_cliente = [-1] * numero_clientes
    capacidad_restante_instalacion = list(capacidad_instalacion)

    orden_clientes = sorted(
        range(numero_clientes),
        key=lambda j: demanda_cliente[j],
        reverse=True,
    )

    for indice_cliente in orden_clientes:

        dem = demanda_cliente[indice_cliente]
        inc = incompatibles_cliente[indice_cliente] if incompatibles_cliente else set()

        costos_m = []

        for indice_instalacion in range(numero_instalaciones):

            cabe = (
                capacidad_restante_instalacion[indice_instalacion]
                >= dem
            )

            if not cabe:
                costos_m.append(float("inf"))
                continue

            if any(
                instalacion_asignada_al_cliente[k] == indice_instalacion
                for k in inc
            ):
                costos_m.append(float("inf"))
                continue

            c_env = (
                costo_envio[indice_instalacion][indice_cliente]
                * dem
            )

            if instalacion_esta_abierta[indice_instalacion]:
                costo_marginal = c_env
            else:
                costo_marginal = (
                    c_env
                    + costo_fijo_instalacion[indice_instalacion]
                )

            costos_m.append(costo_marginal)

        factibles = [
            c for c in costos_m
            if c < float("inf")
        ]

        if not factibles:
            raise RuntimeError(
                f"No hay instalación factible para el cliente "
                f"{indice_cliente} (capacidad o incompatibilidades). "
                f"La instancia podría ser infactible."
            )

        c_min = min(factibles)
        c_max = max(factibles)

        umbral = (
            c_min
            + alfa * (c_max - c_min)
        )

        rcl = [
            i for i in range(numero_instalaciones)
            if costos_m[i] <= umbral
        ]

        instalacion_elegida = random.choice(rcl)

        instalacion_asignada_al_cliente[indice_cliente] = instalacion_elegida
        capacidad_restante_instalacion[instalacion_elegida] -= (
            dem
        )
        instalacion_esta_abierta[instalacion_elegida] = True

    return (
        instalacion_esta_abierta,
        instalacion_asignada_al_cliente,
        capacidad_restante_instalacion,
    )

### Búsqueda local completa

In [385]:
def mejorar_solucion_con_busqueda_local(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    capacidad_instalacion: list[int],
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]] | None = None,
) -> SolucionCFLP:
    sol: SolucionCFLP = (
        list(instalacion_esta_abierta),
        list(instalacion_asignada_al_cliente),
        list(capacidad_restante_instalacion),
    )

    while True:
        sol, mejora_mov1 = _reasignar_clientes_en_barrido(
            sol[0], sol[1], sol[2],
            numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio,
            incompatibles_cliente,
        )
        sol, mejora_mov2 = _cerrar_instalaciones_poco_rentables(
            sol[0], sol[1], sol[2],
            numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio,
            incompatibles_cliente,
        )
        sol, mejora_mov3 = _abrir_instalaciones_rentables(
            sol[0], sol[1], sol[2],
            numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio,
            incompatibles_cliente,
        )
        sol, mejora_mov4 = _intercambiar_clientes_en_barrido(
            sol[0], sol[1], sol[2],
            numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio,
            incompatibles_cliente,
        )
        if not mejora_mov1 and not mejora_mov2 and not mejora_mov3 and not mejora_mov4:
            break

    return sol

### Ejecución GRASP

Bucle de N iteraciones: construir -> mejorar con BL -> guardar si es mejor.
Si usar_alfa_adaptativo=True, el selector ML actualiza alfa tras cada iteración,
usando las primeras muestras_exploracion como exploración aleatoria.

In [386]:
def ejecutar_grasp(
    numero_instalaciones: int,
    numero_clientes: int,
    capacidad_instalacion: list[int],
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    alfa: float = 0.5,
    numero_maximo_de_iteraciones: int = 100,
    semilla_aleatoria: int = 0,
    usar_alfa_adaptativo: bool = True,
    muestras_exploracion: int = 15,
    pares_incompatibles: list[tuple[int, int]] | None = None,
    tiempo_limite_s: float | None = None,
) -> ResultadoGRASP:
    random.seed(semilla_aleatoria)
    t_inicio = time.perf_counter()

    incompatibles = _precomputar_incompatibles(numero_clientes, pares_incompatibles or [])

    mejor_costo_encontrado: float = float("inf")
    mejor_instalacion_esta_abierta: list[bool] | None = None
    mejor_instalacion_asignada: list[int] | None = None
    mejor_capacidad_restante: list[int] | None = None

    if usar_alfa_adaptativo:
        selector_ml = SelectorAlfaML(min_muestras=muestras_exploracion)

    print("=" * 65)
    print("  GRASP - MS-CFLP-CI")
    print("=" * 65)
    print(f"  Instalaciones      : {numero_instalaciones}")
    print(f"  Clientes           : {numero_clientes}")
    print(f"  Pares incompatibles: {len(pares_incompatibles or [])}")
    if usar_alfa_adaptativo:
        print("  Alfa               : adaptativo (regresión lineal ML)")
        print(f"  Exploración inicial: {muestras_exploracion} iteraciones")
    else:
        print(f"  Alfa (aleatoriedad): {alfa}")
    print(f"  Iteraciones GRASP  : {numero_maximo_de_iteraciones}")
    print("=" * 65)

    for iteracion in range(1, numero_maximo_de_iteraciones + 1):
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break

        alfa_esta_iteracion = (
            selector_ml.seleccionar_alfa() if usar_alfa_adaptativo else alfa
        )

        (
            instalacion_esta_abierta,
            instalacion_asignada_al_cliente,
            capacidad_restante_instalacion,
        ) = construir_solucion_greedy_aleatoria(
            numero_instalaciones,
            numero_clientes,
            capacidad_instalacion,
            costo_fijo_instalacion,
            demanda_cliente,
            costo_envio,
            alfa_esta_iteracion,
            incompatibles,
        )

        costo_construida = calcular_costo_total(
            instalacion_esta_abierta,
            instalacion_asignada_al_cliente,
            costo_fijo_instalacion,
            demanda_cliente,
            costo_envio,
        )

        (
            instalacion_esta_abierta,
            instalacion_asignada_al_cliente,
            capacidad_restante_instalacion,
        ) = mejorar_solucion_con_busqueda_local(
            instalacion_esta_abierta,
            instalacion_asignada_al_cliente,
            capacidad_restante_instalacion,
            numero_instalaciones,
            numero_clientes,
            capacidad_instalacion,
            costo_fijo_instalacion,
            demanda_cliente,
            costo_envio,
            incompatibles,
        )

        costo_mejorada = calcular_costo_total(
            instalacion_esta_abierta,
            instalacion_asignada_al_cliente,
            costo_fijo_instalacion,
            demanda_cliente,
            costo_envio,
        )

        if usar_alfa_adaptativo:
            selector_ml.registrar(alfa_esta_iteracion, costo_mejorada)

        if costo_mejorada < mejor_costo_encontrado:
            mejor_costo_encontrado = costo_mejorada
            mejor_instalacion_esta_abierta = list(instalacion_esta_abierta)
            mejor_instalacion_asignada = list(instalacion_asignada_al_cliente)
            mejor_capacidad_restante = list(capacidad_restante_instalacion)
            indicador_nueva_mejor = " << NUEVA MEJOR"
        else:
            indicador_nueva_mejor = ""

        indicador_alfa = (
            f"alfa={alfa_esta_iteracion:.2f}"
            if usar_alfa_adaptativo
            else f"alfa={alfa:.2f}"
        )
        print(
            f"  Iter {iteracion:>4} | "
            f"{indicador_alfa} | "
            f"Construida: {costo_construida:>8,.0f} | "
            f"Mejorada: {costo_mejorada:>8,.0f} | "
            f"Mejor: {mejor_costo_encontrado:>8,.0f}"
            f"{indicador_nueva_mejor}"
        )

    if usar_alfa_adaptativo:
        alfas_usados, costos_registrados, modelo_final = selector_ml.obtener_estado()
        if modelo_final is not None:
            coefs = modelo_final.coef_
            print(
                f"\n  [ML] Modelo entrenado con "
                f"{len(alfas_usados)} observaciones."
            )
            print(
                f"  [ML] Coeficientes (grado 2): "
                f"{[round(c, 4) for c in coefs]}"
            )
            alfa_optimo = alfas_usados[int(np.argmin(costos_registrados))]
            print(f"  [ML] Mejor alfa observado: {alfa_optimo:.2f}")

    print("=" * 65)
    print(f"  SOLUCIÓN FINAL - Costo total: {mejor_costo_encontrado:,.0f}")
    print("=" * 65)

    return {
        "costo_total_minimo": mejor_costo_encontrado,
        "instalaciones_abiertas": mejor_instalacion_esta_abierta,
        "cliente_asignado_a_instalacion": mejor_instalacion_asignada,
        "capacidad_restante_por_instalacion": mejor_capacidad_restante,
    }

### Mostrar resultados detallados

In [387]:
def mostrar_resultados_detallados(
    resultado_grasp: ResultadoGRASP,
    capacidad_instalacion: list[int],
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
) -> None:
    instalaciones_abiertas = resultado_grasp["instalaciones_abiertas"]
    cliente_asignado = resultado_grasp["cliente_asignado_a_instalacion"]
    capacidad_restante = resultado_grasp[
        "capacidad_restante_por_instalacion"
    ]
    costo_total_minimo = resultado_grasp["costo_total_minimo"]

    numero_instalaciones = len(instalaciones_abiertas)
    numero_clientes = len(cliente_asignado)

    print("\n" + "=" * 65)
    print("  DETALLE DE INSTALACIONES ABIERTAS")
    print("=" * 65)

    suma_costos_fijos = 0
    suma_costos_de_envio = 0

    for i in range(numero_instalaciones):

        if not instalaciones_abiertas[i]:
            continue

        clientes_de_esta = [
            j for j in range(numero_clientes)
            if cliente_asignado[j] == i
        ]

        capacidad_usada = (
            capacidad_instalacion[i] - capacidad_restante[i]
        )

        costo_fijo_de_esta = costo_fijo_instalacion[i]
        suma_costos_fijos += costo_fijo_de_esta

        costo_envio_de_esta = sum(
            demanda_cliente[j] * costo_envio[i][j]
            for j in clientes_de_esta
        )
        suma_costos_de_envio += costo_envio_de_esta

        print(
            f"  Instalación {i:>3} | "
            f"Costo fijo: {costo_fijo_de_esta:>4} | "
            f"Capacidad: {capacidad_usada:>3}/{capacidad_instalacion[i]:>3}"
            f" | Clientes: {len(clientes_de_esta):>3} | "
            f"Costo envíos: {costo_envio_de_esta:>7,.0f}"
        )

    print("-" * 65)
    print(f"  Total instalaciones abiertas : {sum(instalaciones_abiertas)}")
    print(f"  Costo fijo total             : {suma_costos_fijos:>10,.0f}")
    print(f"  Costo de envíos total        : {suma_costos_de_envio:>10,.0f}")
    print(f"  COSTO TOTAL                  : {costo_total_minimo:>10,.0f}")
    print("=" * 65)


### GRASP + ML: selector adaptativo de alfa (regresión polinómica)

Ajusta un polinomio de grado configurable sobre el historial de pares (alfa, coste normalizado)
observados en iteraciones anteriores y elige el alfa que minimiza el coste predicho.
El grado se selecciona empíricamente antes de ejecutar las comparativas (véase la celda de comparación de grados).

**Diseño en dos fases** dentro del mismo presupuesto de *N* iteraciones:

1. **Exploración** (primeras `min_muestras` iteraciones): alfa aleatorio uniforme en [0, 1],
   sin modelo, para obtener puntos de entrenamiento distribuidos.
2. **Explotación** (iteraciones restantes): elige el alfa con menor coste predicho
   por el polinomio.


In [388]:
from scipy.optimize import minimize_scalar

class SelectorAlfaML:

    def __init__(
        self, grado: int, min_muestras: int = 20
    ) -> None:
        self.min_muestras = min_muestras
        self.grado = grado
        self.historial_alfa: list[float] = []
        self.historial_costo: list[float] = []
        self.poly = PolynomialFeatures(degree=grado, include_bias=False)
        self.modelo_actual: LinearRegression | None = None

    def registrar(self, alfa: float, costo: float) -> None:
        self.historial_alfa.append(alfa)
        self.historial_costo.append(costo)

    def _entrenar_modelo(self, X, y) -> LinearRegression:
        return LinearRegression().fit(X, y)

    def _predecir(self, modelo: LinearRegression, X_cand) -> np.ndarray:
        return modelo.predict(X_cand)

    def seleccionar_alfa(self) -> float:
        n = len(self.historial_alfa)

        if n < self.min_muestras:
            return round(random.uniform(0.00, 1.00), 4)

        X = self.poly.fit_transform(
            np.array(self.historial_alfa, dtype=float).reshape(-1, 1)
        )
        costos = np.array(self.historial_costo, dtype=float)
        cmin, cmax = costos.min(), costos.max()
        y_norm = (costos - cmin) / (cmax - cmin) if cmax > cmin else np.zeros_like(costos)

        modelo = self._entrenar_modelo(X, y_norm)
        self.modelo_actual = modelo

        def _objetivo(alfa: float) -> float:
            X_a = self.poly.transform(np.array([[alfa]]))
            return float(self._predecir(modelo, X_a)[0])

        resultado = minimize_scalar(_objetivo, bounds=(0.01, 0.99), method="bounded")
        return round(float(resultado.x), 4)

    def obtener_estado(
        self,
    ) -> tuple[list[float], list[float], LinearRegression | None]:
        return (
            list(self.historial_alfa),
            list(self.historial_costo),
            self.modelo_actual,
        )


In [389]:
class SelectorAlfaMLTemporizador(SelectorAlfaML):

    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.tiempo_entrenamiento_s: float = 0.0
        self.tiempo_inferencia_s: float = 0.0

    def _entrenar_modelo(self, X, y) -> LinearRegression:
        t0 = time.perf_counter()
        modelo = super()._entrenar_modelo(X, y)
        self.tiempo_entrenamiento_s += time.perf_counter() - t0
        return modelo

    def _predecir(self, modelo: LinearRegression, X_cand) -> np.ndarray:
        t0 = time.perf_counter()
        pred = super()._predecir(modelo, X_cand)
        self.tiempo_inferencia_s += time.perf_counter() - t0
        return pred


def _suprimir_stdout(fn, *args, **kwargs):
    _stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        result = fn(*args, **kwargs)
    finally:
        sys.stdout = _stdout
    return result



In [390]:
class SelectorAlfaUCB:

    ALFAS = [0.1, 0.3, 0.5, 0.7, 0.9]

    def __init__(self, c: float = 1.0) -> None:
        self.c = c
        self.conteo = [0] * len(self.ALFAS)
        self.suma_recompensa = [0.0] * len(self.ALFAS)
        self.total_tiradas = 0
        self.ultimo_brazo: int | None = None
        self.historial_alfa: list[float] = []
        self.historial_costo: list[float] = []
        self.tiempo_entrenamiento_s = 0.0
        self.tiempo_inferencia_s = 0.0

    def seleccionar_alfa(self) -> float:
        t0 = time.perf_counter()
        no_probados = [i for i, n in enumerate(self.conteo) if n == 0]
        if no_probados:
            indice = no_probados[0]
        else:
            ucb = [
                self.suma_recompensa[i] / self.conteo[i]
                + self.c * math.sqrt(2 * math.log(self.total_tiradas) / self.conteo[i])
                for i in range(len(self.ALFAS))
            ]
            indice = max(range(len(self.ALFAS)), key=lambda i: ucb[i])
        self.ultimo_brazo = indice
        self.tiempo_inferencia_s += time.perf_counter() - t0
        return self.ALFAS[indice]

    def registrar(self, alfa: float, costo: float) -> None:
        t0 = time.perf_counter()
        self.historial_alfa.append(alfa)
        self.historial_costo.append(costo)

        cmin, cmax = min(self.historial_costo), max(self.historial_costo)
        recompensa = (cmax - costo) / (cmax - cmin) if cmax > cmin else 1.0

        indice = self.ultimo_brazo
        self.conteo[indice] += 1
        self.suma_recompensa[indice] += recompensa
        self.total_tiradas += 1
        self.tiempo_entrenamiento_s += time.perf_counter() - t0



class SelectorAlfaThompson:

    ALFAS = [0.1, 0.3, 0.5, 0.7, 0.9]

    def __init__(self) -> None:
        self.a = [1.0] * len(self.ALFAS)
        self.b = [1.0] * len(self.ALFAS)
        self.conteo = [0] * len(self.ALFAS)
        self.total_tiradas = 0
        self.ultimo_brazo: int | None = None
        self.historial_alfa: list[float] = []
        self.historial_costo: list[float] = []
        self.tiempo_entrenamiento_s = 0.0
        self.tiempo_inferencia_s = 0.0

    def seleccionar_alfa(self) -> float:
        t0 = time.perf_counter()
        muestras = [random.betavariate(self.a[i], self.b[i]) for i in range(len(self.ALFAS))]
        indice = max(range(len(self.ALFAS)), key=lambda i: muestras[i])
        self.ultimo_brazo = indice
        self.tiempo_inferencia_s += time.perf_counter() - t0
        return self.ALFAS[indice]

    def registrar(self, alfa: float, costo: float) -> None:
        t0 = time.perf_counter()
        self.historial_alfa.append(alfa)
        self.historial_costo.append(costo)

        cmin, cmax = min(self.historial_costo), max(self.historial_costo)
        recompensa = (cmax - costo) / (cmax - cmin) if cmax > cmin else 1.0

        indice = self.ultimo_brazo
        self.a[indice] += recompensa
        self.b[indice] += (1.0 - recompensa)
        self.conteo[indice] += 1
        self.total_tiradas += 1
        self.tiempo_entrenamiento_s += time.perf_counter() - t0

### GRASP + ML: selectores de alfa alternativos (UCB1 y Thompson Sampling)

Ademas del selector por regresion polinomica (arriba), el cuaderno incluye dos
selectores de alfa basados en bandidos multi-brazo sobre un conjunto discreto
de candidatos `{0.1, 0.3, 0.5, 0.7, 0.9}`, en vez de una regresion continua:

- **`SelectorAlfaUCB`**: bandido UCB1. Tras probar cada brazo una vez, elige el
  que maximiza la media observada mas un termino de exploracion que depende del
  numero de veces que se ha probado ese brazo.
- **`SelectorAlfaThompson`**: bandido bayesiano. Cada brazo se modela con una
  distribucion Beta sobre su recompensa esperada; en cada iteracion se
  muestrea un valor de cada Beta y se elige el brazo con la muestra mas alta.

`ejecutar_grasp_ml` selecciona entre los tres selectores (regresion polinomica,
UCB1, Thompson) mediante los parametros `usar_ucb1` y `usar_thompson`.

### GRASP + ML: ejecución con selector polinómico

Bucle de `n_iter` iteraciones con el mismo presupuesto que GRASP base.
El selector polinómico sustituye el alfa fijo en dos fases:

- **Exploración** (`rand`): las primeras `min_muestras` iteraciones seleccionan $\alpha$ uniformemente al azar para acumular el historial $(\alpha_t, C_t)$ mínimo que necesita el ajuste del modelo.
- **Explotación** (`ML`): a partir de la muestra `min_muestras`, ajusta la regresión polinómica sobre el historial y selecciona el $\alpha$ que minimiza $\hat{f}$ en $[0.01, 0.99]$ mediante minimización continua (método de Brent acotado).

> El grado del polinomio **no está fijado aquí**: debe pasarse como argumento `grado` y en las comparativas se usa `_MEJOR_GRADO`, el grado que minimizó el gap medio en la comparación de grados previa.
> `min_muestras=15` garantiza un ajuste mínimamente estable antes de explotar el modelo; por tanto `n_iter` debe ser al menos `min_muestras + 1`.

In [391]:
def _reconstruir_estado_pr(asignada, num_inst, num_cli, cap, dem):
    abierta = [False] * num_inst
    restante = list(cap)
    for j in range(num_cli):
        i = asignada[j]
        restante[i] -= dem[j]
        abierta[i] = True
    return abierta, restante


def _factible_mover_pr(cliente, destino, asignada, restante, dem, incompatibles):
    if restante[destino] < dem[cliente]:
        return False
    for k in incompatibles[cliente]:
        if asignada[k] == destino:
            return False
    return True


def path_relinking(sol_inicial, sol_guia, num_instalaciones, num_clientes,
                   capacidad_instalacion, costo_fijo_instalacion, demanda_cliente,
                   costo_envio, incompatibles_cliente):
    asig = list(sol_inicial[1])
    abierta, restante = _reconstruir_estado_pr(asig, num_instalaciones, num_clientes,
                                               capacidad_instalacion, demanda_cliente)
    asig_guia = sol_guia[1]

    carga = [0] * num_instalaciones
    for j in range(num_clientes):
        carga[asig[j]] += 1

    costo_actual = calcular_costo_total(abierta, asig, costo_fijo_instalacion,
                                        demanda_cliente, costo_envio)
    costo_guia = calcular_costo_total(sol_guia[0], asig_guia, costo_fijo_instalacion,
                                      demanda_cliente, costo_envio)
    if costo_actual <= costo_guia:
        mejor_asig, mejor_costo = list(asig), costo_actual
    else:
        mejor_asig, mejor_costo = list(asig_guia), costo_guia

    difieren = {j for j in range(num_clientes) if asig[j] != asig_guia[j]}

    while difieren:
        mejor_mov = None
        for j in difieren:
            destino = asig_guia[j]
            if not _factible_mover_pr(j, destino, asig, restante, demanda_cliente, incompatibles_cliente):
                continue
            origen = asig[j]
            d = demanda_cliente[j]
            envio_delta = (costo_envio[destino][j] - costo_envio[origen][j]) * d
            apertura = costo_fijo_instalacion[destino] if not abierta[destino] else 0
            cierre = -costo_fijo_instalacion[origen] if carga[origen] == 1 else 0
            costo_result = costo_actual + envio_delta + apertura + cierre
            if mejor_mov is None or costo_result < mejor_mov[0]:
                mejor_mov = (costo_result, j, destino)

        if mejor_mov is None:
            break

        costo_result, j, destino = mejor_mov
        origen = asig[j]
        d = demanda_cliente[j]
        restante[origen] += d
        restante[destino] -= d
        carga[origen] -= 1
        carga[destino] += 1
        asig[j] = destino
        abierta[destino] = True
        if carga[origen] == 0:
            abierta[origen] = False
        costo_actual = costo_result
        difieren.discard(j)

        if costo_actual < mejor_costo:
            mejor_costo = costo_actual
            mejor_asig = list(asig)

    ab_final, rest_final = _reconstruir_estado_pr(mejor_asig, num_instalaciones, num_clientes,
                                                  capacidad_instalacion, demanda_cliente)
    return ab_final, mejor_asig, rest_final, mejor_costo


In [392]:
def ejecutar_grasp_ml(
    datos: DatosCFLP,
    grado: int = 2,
    n_iter: int = 50,
    min_muestras: int = 15,
    semilla: int = 0,
    verbose: bool = True,
    tiempo_limite_s: float | None = None,
    usar_pr: bool = True,
    tam_elite: int = 5,
    usar_ucb1: bool = False,
    usar_thompson: bool = False,
) -> "ResultadoBenchmark":
    random.seed(semilla)
    F = datos["numero_instalaciones"]
    C = datos["numero_clientes"]
    cap = datos["capacidad_instalacion"]
    cf = datos["costo_fijo_instalacion"]
    dem = datos["demanda_cliente"]
    envio = datos["costo_envio"]
    incompatibles = _precomputar_incompatibles(C, datos["pares_incompatibles"])
    selector = (
        SelectorAlfaThompson() if usar_thompson
        else SelectorAlfaUCB() if usar_ucb1
        else SelectorAlfaML(min_muestras=min_muestras, grado=grado)
    )
    mejor_costo = float("inf")
    t_train = 0.0
    t_infer = 0.0
    t_inicio = time.perf_counter()

    elite: list[tuple[float, SolucionCFLP]] = []

    def _intentar_insertar_elite(sol, costo):
        for c, _ in elite:
            if c == costo:
                return
        elite.append((costo, (list(sol[0]), list(sol[1]), list(sol[2]))))
        elite.sort(key=lambda t: t[0])
        del elite[tam_elite:]

    if verbose:
        print("=" * 60)
        etiqueta = "GRASP + ML + PR" if usar_pr else "GRASP + ML"
        metodo = (
            "Thompson Sampling" if usar_thompson
            else "bandido UCB1" if usar_ucb1
            else f"regresión polinómica grado {grado}"
        )
        print(f"  {etiqueta}  ({metodo})")
        print("=" * 60)
        print(f"  {'Iter':>4}  {'Fase':<6}  {'Alfa':>6}  {'Coste':>12}  {'Mejor':>12}")
        print(f"  {'-'*50}")

    for it in range(1, n_iter + 1):
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break

        _t0 = time.perf_counter()
        alfa = selector.seleccionar_alfa()
        t_infer += time.perf_counter() - _t0

        fase = (
            "TS" if usar_thompson
            else "UCB1" if usar_ucb1
            else ("rand" if len(selector.historial_alfa) < min_muestras else "ML")
        )

        ab, asig, rest = construir_solucion_greedy_aleatoria(
            F, C, cap, cf, dem, envio, alfa, incompatibles,
        )
        ab, asig, rest = mejorar_solucion_con_busqueda_local(
            ab, asig, rest, F, C, cap, cf, dem, envio, incompatibles,
        )
        costo = calcular_costo_total(ab, asig, cf, dem, envio)
        sol = (ab, asig, rest)

        if usar_pr and elite:
            _, sol_elite = random.choice(elite)
            ab_pr, asig_pr, rest_pr, costo_pr = path_relinking(
                sol, sol_elite, F, C, cap, cf, dem, envio, incompatibles,
            )
            ab_pr, asig_pr, rest_pr = mejorar_solucion_con_busqueda_local(
                ab_pr, asig_pr, rest_pr, F, C, cap, cf, dem, envio, incompatibles,
            )
            costo_pr = calcular_costo_total(ab_pr, asig_pr, cf, dem, envio)
            if costo_pr < costo:
                sol, costo = (ab_pr, asig_pr, rest_pr), costo_pr

        _t0 = time.perf_counter()
        selector.registrar(alfa, costo)
        t_train += time.perf_counter() - _t0

        if usar_pr:
            _intentar_insertar_elite(sol, costo)

        if costo < mejor_costo:
            mejor_costo = costo

        if verbose:
            print(f"  {it:>4}  {fase:<6}  {alfa:>6.2f}  {costo:>12,.0f}  {mejor_costo:>12,.0f}")

    t_total = time.perf_counter() - t_inicio
    if verbose:
        print(f"  {'-'*50}")
        print(f"  Mejor coste: {mejor_costo:,.0f}")
        print(f"  Tiempo total: {t_total:.2f}s"
              f"  (ML entreno: {t_train:.3f}s  inferencia: {t_infer:.3f}s)")

    if usar_thompson:
        nombre = "GRASP+Thompson+PR" if usar_pr else "GRASP+Thompson"
    elif usar_ucb1:
        nombre = "GRASP+UCB1+PR" if usar_pr else "GRASP+UCB1"
    else:
        nombre = f"GRASP+ML+PR(g{grado})" if usar_pr else f"GRASP+ML(g{grado})"
    return ResultadoBenchmark(
        nombre=nombre,
        costo_final=mejor_costo,
        tiempo_total_s=t_total,
        tiempo_entrenamiento_ml_s=t_train,
        tiempo_inferencia_ml_s=t_infer,
        modelo_ml=selector,
    )


### Evaluacion de los selectores de alfa

Esta celda genera pares (alfa, coste) sobre las instancias `wlp01`-`wlp04` para
ajustar y comparar distintos grados del selector polinomico.

#### Que se mide

Para cada instancia se generan **N = 20 pares (alfa, coste)** ejecutando GRASP + BL
con alfas aleatorios. Sobre esas muestras se calculan:

- **R^2 LOO** (*leave-one-out*).
- **RMSE y MAE LOO**.
- **|Delta alfa*|**: diferencia entre el alfa observado y el predicho por el modelo.

In [393]:
import time as _time
import os
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

INSTANCIAS_EVAL = ["wlp01.dzn", "wlp02.dzn", "wlp03.dzn", "wlp04.dzn"]
DATA_DIR        = "Instaces_MS-CFLP-CI/Public"
N_MUESTRAS      = 20
SEMILLA         = 42

def _grasp_con_alfa(datos, alfa, semilla_local=0):
    incompatibles = _precomputar_incompatibles(
        datos["numero_clientes"], datos["pares_incompatibles"]
    )
    random.seed(semilla_local)
    ab, asig, rest = construir_solucion_greedy_aleatoria(
        datos["numero_instalaciones"], datos["numero_clientes"],
        datos["capacidad_instalacion"], datos["costo_fijo_instalacion"],
        datos["demanda_cliente"], datos["costo_envio"], alfa, incompatibles,
    )
    ab, asig, rest = mejorar_solucion_con_busqueda_local(
        ab, asig, rest,
        datos["numero_instalaciones"], datos["numero_clientes"],
        datos["capacidad_instalacion"], datos["costo_fijo_instalacion"],
        datos["demanda_cliente"], datos["costo_envio"], incompatibles,
    )
    return calcular_costo_total(
        ab, asig, datos["costo_fijo_instalacion"],
        datos["demanda_cliente"], datos["costo_envio"],
    )

MODELOS = {
    "Poly-1": ("poly", 1),
    "Poly-2": ("poly", 2),
    "Poly-3": ("poly", 3),
}

def _ajustar_y_predecir(nombre, X_tr, y_tr, X_te):
    kind, param = MODELOS[nombre]
    if kind == "poly":
        pf  = PolynomialFeatures(param, include_bias=True)
        reg = LinearRegression()
        reg.fit(pf.fit_transform(X_tr), y_tr)
        return reg.predict(pf.transform(X_te))

def _loo_cv(nombre, X, y):
    kind, param = MODELOS[nombre]
    n = len(X)
    if kind == "poly" and n < param + 2:
        return np.nan, np.nan, np.nan
    loo    = LeaveOneOut()
    y_pred = np.empty(n)
    for tr_idx, te_idx in loo.split(X):
        try:
            y_pred[te_idx] = _ajustar_y_predecir(nombre, X[tr_idx], y[tr_idx], X[te_idx])
        except Exception:
            y_pred[te_idx] = np.nan
    mask = ~np.isnan(y_pred)
    if mask.sum() < 2:
        return np.nan, np.nan, np.nan
    r2   = r2_score(y[mask], y_pred[mask])
    rmse = np.sqrt(mean_squared_error(y[mask], y_pred[mask]))
    mae  = mean_absolute_error(y[mask], y_pred[mask])
    return r2, rmse, mae

def _alfa_optima_pred(nombre, X_all, y_all):
    candidatos = np.linspace(0.0, 1.0, 200).reshape(-1, 1)
    preds = _ajustar_y_predecir(nombre, X_all, y_all, candidatos)
    return float(candidatos[np.argmin(preds)])

_EJECUTAR_EVALUACION_LOO = False

if _EJECUTAR_EVALUACION_LOO:
    TAMANIOS_LC = [3, 5, 7, 10, 15, 20]
    
    metricas_global = {nm: {"r2": [], "rmse": [], "mae": []} for nm in MODELOS}
    lc_global       = {nm: {k: [] for k in TAMANIOS_LC} for nm in MODELOS}
    
    for archivo in INSTANCIAS_EVAL:
        nombre_inst = archivo.replace(".dzn", "")
        datos = leer_instancia_dzn(os.path.join(DATA_DIR, archivo))
        F, C  = datos["numero_instalaciones"], datos["numero_clientes"]
    
        t0 = _time.perf_counter()
        np.random.seed(SEMILLA); random.seed(SEMILLA)
        alfas  = np.array([round(random.uniform(0.0, 1.0), 2) for _ in range(N_MUESTRAS)])
        costos = np.array([_grasp_con_alfa(datos, a, SEMILLA + i) for i, a in enumerate(alfas)])
        t_gen  = _time.perf_counter() - t0
    
        X = alfas.reshape(-1, 1)
        y = costos
        alfa_obs = alfas[np.argmin(costos)]
    
        print(f"\n{'='*62}")
        print(f"  {nombre_inst}  ({F} almacenes, {C} tiendas)  — muestras en {t_gen:.1f}s")
        print(f"  α* observado = {alfa_obs:.2f}  |  coste mín observado = {costos.min():.0f}")
        print(f"\n  {'Modelo':<9} {'R²(LOO)':>9} {'RMSE(LOO)':>11} {'MAE(LOO)':>10}"
              f" {'α* pred':>8} {'|Δα*|':>6}")
        print(f"  {'-'*57}")
    
        for nm in MODELOS:
            r2, rmse, mae = _loo_cv(nm, X, y)
            if np.isnan(r2):
                print(f"  {nm:<9} {'N/A':>9} {'N/A':>11} {'N/A':>10}")
                continue
            alfa_pred  = _alfa_optima_pred(nm, X, y)
            delta_alfa = abs(alfa_pred - alfa_obs)
            print(f"  {nm:<9} {r2:>+9.4f} {rmse:>11.0f} {mae:>10.0f}"
                  f" {alfa_pred:>8.2f} {delta_alfa:>6.2f}")
            metricas_global[nm]["r2"].append(r2)
            metricas_global[nm]["rmse"].append(rmse)
            metricas_global[nm]["mae"].append(mae)
    
        for nm in MODELOS:
            for k in TAMANIOS_LC:
                r2k, _, _ = _loo_cv(nm, X[:k], y[:k])
                lc_global[nm][k].append(r2k)
    
    print(f"\n\n{'='*62}")
    print("  RESUMEN GLOBAL — media sobre wlp01–wlp04")
    print(f"  {'Modelo':<9} {'R²(LOO)':>9} {'RMSE(LOO)':>11} {'MAE(LOO)':>10}")
    print(f"  {'-'*41}")
    for nm in MODELOS:
        vals_r2   = metricas_global[nm]["r2"]
        vals_rmse = metricas_global[nm]["rmse"]
        vals_mae  = metricas_global[nm]["mae"]
        if not vals_r2:
            continue
        print(f"  {nm:<9} {np.mean(vals_r2):>+9.4f}"
              f" {np.mean(vals_rmse):>11.0f} {np.mean(vals_mae):>10.0f}")
    
    print(f"\n\n  Curva de aprendizaje — R²(LOO) medio (wlp01–wlp04):")
    header_lc = f"  {'n':>4}"
    for nm in MODELOS:
        header_lc += f"  {nm:>9}"
    print(header_lc)
    print(f"  {'-'*(6 + len(MODELOS) * 11)}")
    for k in TAMANIOS_LC:
        row = f"  {k:>4}"
        for nm in MODELOS:
            vals = [v for v in lc_global[nm][k] if v is not None and not np.isnan(v)]
            row += f"  {np.mean(vals):>+9.4f}" if vals else f"  {'N/A':>9}"
        print(row)
    
    print()
    print("  Nota: Poly-d requiere ≥ d+2 muestras para LOO significativo.")
    print("  GRASP+ML usa n=3–5 muestras antes de explotar el modelo Poly-2.")
    
    fig, axes = plt.subplots(1, len(INSTANCIAS_EVAL), figsize=(14, 4), sharey=False)
    colores = {"Poly-1": "tab:blue", "Poly-2": "tab:orange",
               "Poly-3": "tab:green", "RF-50": "tab:red"}
    
    for ax, archivo in zip(axes, INSTANCIAS_EVAL):
        nombre_inst = archivo.replace(".dzn", "")
        datos = leer_instancia_dzn(os.path.join(DATA_DIR, archivo))
        np.random.seed(SEMILLA); random.seed(SEMILLA)
        alfas_p  = np.array([round(random.uniform(0.0, 1.0), 2) for _ in range(N_MUESTRAS)])
        costos_p = np.array([_grasp_con_alfa(datos, a, SEMILLA + i) for i, a in enumerate(alfas_p)])
        X_p = alfas_p.reshape(-1, 1)
    
        ax.scatter(alfas_p, costos_p, color="gray", s=20, alpha=0.6, label="muestras")
        grid = np.linspace(0.0, 1.0, 200).reshape(-1, 1)
        for nm, col in colores.items():
            try:
                preds = _ajustar_y_predecir(nm, X_p, costos_p, grid)
                ax.plot(grid, preds, color=col, linewidth=1.5, label=nm)
            except Exception:
                pass
        ax.set_title(nombre_inst)
        ax.set_xlabel("alfa (α)")
        ax.set_ylabel("coste")
        ax.legend(fontsize=7)
    
    plt.suptitle("Ajuste de modelos: curva alfa → coste con BL (20 muestras por instancia)")
    plt.tight_layout()
    plt.show()


### Comparacion de grados polinomicos en GRASP+ML

Ejecuta `ejecutar_grasp_ml` con distintos grados $d \in \{1, 2, 3, 4, 5, 6\}$ sobre
`wlp01`-`wlp05` con varias semillas. El resultado (`_MEJOR_GRADO`) se usa en las
comparativas posteriores.

In [394]:
import os as _os2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

_EJECUTAR_COMPARATIVA = False
_MEJOR_GRADO_DEFAULT  = 2

_INSTANCIAS_GRADO = ["wlp01.dzn", "wlp02.dzn", "wlp03.dzn", "wlp04.dzn"]
_REFERENCIA_GRADO = {"wlp01": 28716, "wlp02": 52952, "wlp03": 64296, "wlp04": 84633}
_GRADOS     = [1, 2, 3, 4, 5, 6]
_N_ITER_G   = 50
_MIN_M_G    = 15
_SEMILLAS_G = [0, 1, 2, 3, 4]

if not _EJECUTAR_COMPARATIVA:
    _MEJOR_GRADO = _MEJOR_GRADO_DEFAULT
    print(f"  [comparativa omitida] _MEJOR_GRADO = {_MEJOR_GRADO} (por defecto)")
else:
    _total = len(_SEMILLAS_G) * len(_INSTANCIAS_GRADO) * len(_GRADOS)
    _done  = 0

    _res = {
        (arch.replace(".dzn", ""), g): {"gaps": [], "tiempos": []}
        for arch in _INSTANCIAS_GRADO
        for g in _GRADOS
    }

    for _sem in _SEMILLAS_G:
        print(f"\n  Semilla {_sem} / {_SEMILLAS_G[-1]}")
        for archivo in _INSTANCIAS_GRADO:
            nombre = archivo.replace(".dzn", "")
            datos  = leer_instancia_dzn(_os2.path.join("Instaces_MS-CFLP-CI/Public", archivo))
            ref    = _REFERENCIA_GRADO[nombre]
            for g in _GRADOS:
                _done += 1
                pct = 100 * _done / _total
                print(f"    [{pct:5.1f}%] {nombre}  Poly-{g}...", end=" ", flush=True)
                rb  = ejecutar_grasp_ml(datos, grado=g, n_iter=_N_ITER_G,
                                        min_muestras=_MIN_M_G, semilla=_sem, verbose=False)
                gap = 100 * (rb.costo_final - ref) / ref
                _res[(nombre, g)]["gaps"].append(gap)
                _res[(nombre, g)]["tiempos"].append(rb.tiempo_total_s)
                print(f"gap {gap:+.2f}%  ({rb.tiempo_total_s:.1f}s)")

    instancias = [a.replace(".dzn", "") for a in _INSTANCIAS_GRADO]
    print()
    print(f"  {'Instancia':<10}", end="")
    for g in _GRADOS:
        print(f" │ {'Poly-'+str(g):^21}", end="")
    print()
    print(f"  {'':10}", end="")
    for g in _GRADOS:
        print(f" │ {'Gap% (media±std)':>14} {'t(s)':>5}", end="")
    print()
    print(f"  {'-'*10}", end="")
    for _ in _GRADOS:
        print(f" ┼ {'-'*14} {'-'*5}", end="")
    print()

    avg_por_grado = {}
    for nombre in instancias:
        print(f"  {nombre:<10}", end="")
        for g in _GRADOS:
            gaps   = _res[(nombre, g)]["gaps"]
            mu_gap = np.mean(gaps);  sd_gap = np.std(gaps)
            mu_t   = np.mean(_res[(nombre, g)]["tiempos"])
            print(f" │ {mu_gap:>+6.2f}±{sd_gap:<5.2f}% {mu_t:>4.1f}s", end="")
        print()

    print(f"  {'Media':<10}", end="")
    for g in _GRADOS:
        all_gaps = [gap for n in instancias for gap in _res[(n, g)]["gaps"]]
        avg = np.mean(all_gaps)
        avg_por_grado[g] = avg
        print(f" │ {avg:>+6.2f}±{'':5}  {'':5}", end="")
    print()

    _MEJOR_GRADO = min(avg_por_grado, key=avg_por_grado.get)
    print(f"\n  Mejor grado: Poly-{_MEJOR_GRADO} (gap medio {avg_por_grado[_MEJOR_GRADO]:+.2f}%)")
    print(f"  → Se usará Poly-{_MEJOR_GRADO} en las comparaciones posteriores")
    print(f"  Semillas: {_SEMILLAS_G}  |  {_N_ITER_G} iter  |  {_MIN_M_G} min_muestras")

    _cmap   = plt.get_cmap("tab10")
    _colors = {g: _cmap(i) for i, g in enumerate(_GRADOS)}

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(
        f"Comparación de grados polinómicos · {len(_SEMILLAS_G)} semillas · {_N_ITER_G} iter ({_MIN_M_G} expl + {_N_ITER_G - _MIN_M_G} exploit)",
        fontsize=13, fontweight="bold",
    )

    bp_data = [[gap for n in instancias for gap in _res[(n, g)]["gaps"]] for g in _GRADOS]
    bp = axes[0].boxplot(bp_data, patch_artist=True,
                         medianprops={"color": "black", "linewidth": 1.8})
    for patch, g in zip(bp["boxes"], _GRADOS):
        patch.set_facecolor(_colors[g])
        patch.set_alpha(0.85)
        if g == _MEJOR_GRADO:
            patch.set_linewidth(2.5)
            patch.set_edgecolor("black")
    axes[0].set_xticks(range(1, len(_GRADOS) + 1))
    axes[0].set_xticklabels([f"Poly-{g}" for g in _GRADOS])
    axes[0].set_title("Distribución de gap% por grado\n(todas las instancias × semillas)")
    axes[0].set_ylabel("Gap (%)")
    axes[0].axhline(avg_por_grado[_MEJOR_GRADO], color="red", linewidth=1,
                    linestyle="--", alpha=0.6, label=f"Media mínima (Poly-{_MEJOR_GRADO})")
    axes[0].legend(fontsize=8)

    for nombre in instancias:
        mu_gaps = [np.mean(_res[(nombre, g)]["gaps"]) for g in _GRADOS]
        axes[1].plot(_GRADOS, mu_gaps, marker="o", label=nombre, linewidth=1.8)
    mu_global = [avg_por_grado[g] for g in _GRADOS]
    axes[1].plot(_GRADOS, mu_global, marker="D", color="black", linewidth=2.5,
                 linestyle="--", label="Media global", zorder=5)
    axes[1].axvline(_MEJOR_GRADO, color="red", linewidth=1, linestyle=":", alpha=0.7)
    axes[1].set_xticks(_GRADOS)
    axes[1].set_xticklabels([f"Poly-{g}" for g in _GRADOS])
    axes[1].set_title("Gap% medio por grado e instancia")
    axes[1].set_ylabel("Gap medio (%)")
    axes[1].set_xlabel("Grado del polinomio")
    axes[1].legend(fontsize=8)

    mu_g  = [np.mean([gap for n in instancias for gap in _res[(n, g)]["gaps"]]) for g in _GRADOS]
    sd_g  = [np.std( [gap for n in instancias for gap in _res[(n, g)]["gaps"]]) for g in _GRADOS]
    bars = axes[2].bar(_GRADOS, mu_g, yerr=sd_g, capsize=5,
                       color=[_colors[g] for g in _GRADOS], alpha=0.88, width=0.6)
    for bar, g in zip(bars, _GRADOS):
        if g == _MEJOR_GRADO:
            bar.set_edgecolor("black")
            bar.set_linewidth(2.5)
    axes[2].set_xticks(_GRADOS)
    axes[2].set_xticklabels([f"Poly-{g}" for g in _GRADOS])
    axes[2].set_title("Gap% medio global por grado\n(± std entre instancias × semillas)")
    axes[2].set_ylabel("Gap medio (%)")
    axes[2].set_xlabel("Grado del polinomio")
    best_val = avg_por_grado[_MEJOR_GRADO]
    axes[2].annotate(f"mejor\nPoly-{_MEJOR_GRADO}\n{best_val:+.2f}%",
                     xy=(_MEJOR_GRADO, best_val),
                     xytext=(_MEJOR_GRADO + 0.6, best_val + max(sd_g) * 0.5),
                     arrowprops={"arrowstyle": "->", "color": "black"},
                     fontsize=8, ha="left")

    plt.tight_layout()
    _fig_path = "grasp_poly_grados.png"
    plt.savefig(_fig_path, dpi=150, bbox_inches="tight")
    plt.close()
    from IPython.display import Image as _Img, display as _disp
    _disp(_Img(_fig_path))
    print(f"  Figura guardada en {_fig_path}")


  [comparativa omitida] _MEJOR_GRADO = 2 (por defecto)


### Benchmark GRASP (alfa fijo, base) vs GRASP+ML

Compara **GRASP** con $\alpha=0.5$ fijo (`ejecutar_grasp`, `usar_alfa_adaptativo=False`)
contra **GRASP+ML** con el selector polinómico de grado `_MEJOR_GRADO`
(`ejecutar_grasp_ml`), sobre `wlp01`-`wlp04`, 5 semillas por instancia, 50
iteraciones GRASP para ambos (15 de exploración inicial en la variante ML,
igual que en la comparación de grados polinómicos de más arriba).


In [395]:
import os as _os_gab
import time as _tgab
import io as _io_gab
import contextlib as _ctx_gab
import numpy as np

_EJECUTAR_BENCHMARK_GRASP_ML = False

if _EJECUTAR_BENCHMARK_GRASP_ML:

    _INST_GRASP_BENCH    = ["wlp01.dzn", "wlp02.dzn", "wlp03.dzn", "wlp04.dzn"]
    _REF_GRASP_BENCH      = {"wlp01": 28716, "wlp02": 52952, "wlp03": 64296, "wlp04": 84633}
    _SEMILLAS_GRASP_BENCH = [0, 1, 2, 3, 4]
    _N_ITER_GRASP_BENCH   = 50
    _MIN_MUESTRAS_GRASP   = 15

    _grasp_res = {
        nombre: {"GRASP": {"gaps": [], "t": []},
                 "GRASP+ML": {"gaps": [], "t": []}}
        for nombre in [a.replace(".dzn", "") for a in _INST_GRASP_BENCH]
    }

    for archivo in _INST_GRASP_BENCH:
        nombre = archivo.replace(".dzn", "")
        datos  = leer_instancia_dzn(_os_gab.path.join("Instaces_MS-CFLP-CI/Public", archivo))
        ref    = _REF_GRASP_BENCH[nombre]

        print(f"\n  {nombre}")
        for semilla in _SEMILLAS_GRASP_BENCH:
            t0 = _tgab.perf_counter()
            with _ctx_gab.redirect_stdout(_io_gab.StringIO()):
                r_base = ejecutar_grasp(
                    datos["numero_instalaciones"], datos["numero_clientes"],
                    datos["capacidad_instalacion"], datos["costo_fijo_instalacion"],
                    datos["demanda_cliente"], datos["costo_envio"],
                    alfa=0.5, numero_maximo_de_iteraciones=_N_ITER_GRASP_BENCH,
                    semilla_aleatoria=semilla, usar_alfa_adaptativo=False,
                    pares_incompatibles=datos["pares_incompatibles"],
                )
            t_base   = _tgab.perf_counter() - t0
            gap_base = 100 * (r_base["costo_total_minimo"] - ref) / ref
            _grasp_res[nombre]["GRASP"]["gaps"].append(gap_base)
            _grasp_res[nombre]["GRASP"]["t"].append(t_base)

            r_ml = ejecutar_grasp_ml(
                datos, grado=_MEJOR_GRADO, n_iter=_N_ITER_GRASP_BENCH,
                min_muestras=_MIN_MUESTRAS_GRASP, semilla=semilla, verbose=False,
            )
            gap_ml = 100 * (r_ml.costo_final - ref) / ref
            _grasp_res[nombre]["GRASP+ML"]["gaps"].append(gap_ml)
            _grasp_res[nombre]["GRASP+ML"]["t"].append(r_ml.tiempo_total_s)

            print(f"    seed={semilla}  GRASP: {gap_base:+.2f}% {t_base:.1f}s  "
                  f"GRASP+ML: {gap_ml:+.2f}% {r_ml.tiempo_total_s:.1f}s  "
                  f"Δgap={gap_ml-gap_base:+.2f}pp")

    instancias_grasp = [a.replace(".dzn", "") for a in _INST_GRASP_BENCH]
    print(f"\n{'='*70}")
    print(f"  {'':<10}  {'── GRASP base ──':^18}  {'── GRASP+ML ──':^18}  {'Δgap':>6}")
    print(f"  {'Inst.':<10}  {'gap%':>7}  {'σ':>5}  {'t(s)':>5}  "
          f"{'gap%':>7}  {'σ':>5}  {'t(s)':>5}  {'(pp)':>6}")
    print(f"  {'-'*70}")

    all_base, all_ml = [], []
    for nombre in instancias_grasp:
        bg = np.mean(_grasp_res[nombre]["GRASP"]["gaps"])
        bs = np.std(_grasp_res[nombre]["GRASP"]["gaps"])
        bt = np.mean(_grasp_res[nombre]["GRASP"]["t"])
        mg = np.mean(_grasp_res[nombre]["GRASP+ML"]["gaps"])
        ms = np.std(_grasp_res[nombre]["GRASP+ML"]["gaps"])
        mt = np.mean(_grasp_res[nombre]["GRASP+ML"]["t"])
        all_base.append(bg); all_ml.append(mg)
        print(f"  {nombre:<10}  {bg:>+7.2f}  {bs:>5.2f}  {bt:>5.1f}  "
              f"{mg:>+7.2f}  {ms:>5.2f}  {mt:>5.1f}  {mg-bg:>+6.2f}")

    print(f"  {'-'*70}")
    mb = np.mean(all_base); mm = np.mean(all_ml)
    print(f"  {'Media':<10}  {mb:>+7.2f}  {'':<5}  {'':<5}  {mm:>+7.2f}  {'':<5}  {'':<5}  {mm-mb:>+6.2f}")
    print(f"  {'='*70}")
else:
    print("Benchmark GRASP vs GRASP+ML omitido (_EJECUTAR_BENCHMARK_GRASP_ML = False)")


Benchmark GRASP vs GRASP+ML omitido (_EJECUTAR_BENCHMARK_GRASP_ML = False)


### GRASP + Path Relinking

Variante de GRASP que mantiene un **conjunto elite** de soluciones y, tras la
busqueda local de cada iteracion, aplica *path relinking* entre la solucion
recien obtenida y un miembro elite elegido al azar. El path relinking
transforma una solucion en la otra un cliente a la vez, evaluando las
soluciones intermedias del camino. El resultado se refina con busqueda local y
puede entrar al conjunto elite.

In [396]:
from scipy.optimize import minimize_scalar

def ejecutar_grasp_pr(
    datos: DatosCFLP,
    n_iter: int = 50,
    alfa: float = 0.5,
    tam_elite: int = 5,
    semilla: int = 0,
    tiempo_limite_s: float | None = None,
) -> ResultadoBenchmark:
    random.seed(semilla)
    t_inicio = time.perf_counter()
    F = datos["numero_instalaciones"]
    C = datos["numero_clientes"]
    cap = datos["capacidad_instalacion"]
    cf = datos["costo_fijo_instalacion"]
    dem = datos["demanda_cliente"]
    envio = datos["costo_envio"]
    incompatibles = _precomputar_incompatibles(C, datos["pares_incompatibles"])

    elite: list[tuple[float, SolucionCFLP]] = []
    mejor_costo = float("inf")

    def _intentar_insertar_elite(sol, costo):
        for c, _ in elite:
            if c == costo:
                return
        elite.append((costo, (list(sol[0]), list(sol[1]), list(sol[2]))))
        elite.sort(key=lambda t: t[0])
        del elite[tam_elite:]

    for _ in range(n_iter):
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break

        ab, asig, rest = construir_solucion_greedy_aleatoria(
            F, C, cap, cf, dem, envio, alfa, incompatibles,
        )
        ab, asig, rest = mejorar_solucion_con_busqueda_local(
            ab, asig, rest, F, C, cap, cf, dem, envio, incompatibles,
        )
        costo = calcular_costo_total(ab, asig, cf, dem, envio)
        sol = (ab, asig, rest)

        if elite:
            _, sol_elite = random.choice(elite)
            ab_pr, asig_pr, rest_pr, costo_pr = path_relinking(
                sol, sol_elite, F, C, cap, cf, dem, envio, incompatibles,
            )
            ab_pr, asig_pr, rest_pr = mejorar_solucion_con_busqueda_local(
                ab_pr, asig_pr, rest_pr, F, C, cap, cf, dem, envio, incompatibles,
            )
            costo_pr = calcular_costo_total(ab_pr, asig_pr, cf, dem, envio)
            if costo_pr < costo:
                sol, costo = (ab_pr, asig_pr, rest_pr), costo_pr

        _intentar_insertar_elite(sol, costo)
        if costo < mejor_costo:
            mejor_costo = costo

    return ResultadoBenchmark(
        nombre="GRASP+PR",
        costo_final=mejor_costo,
        tiempo_total_s=time.perf_counter() - t_inicio,
    )


## 3. GVNS (General Variable Neighborhood Search)

Alterna perturbación (shaking con radio creciente k) y búsqueda local (VND)
hasta converger. Se usa como refinador sobre la solución de GRASP o la seleccionada por el RF.

```
k <- 1
mientras k <= k_max:
    s' <- Shaking(s, k)
    s'' <- VND(s')
    si coste(s'') < coste(s): s <- s'', k <- 1
    si no:                     k <- k + 1
```

### Estructuras de datos (GVNS)

In [397]:
class ResultadoGVNS(TypedDict):

    costo_total_minimo: float
    instalaciones_abiertas: list[bool]
    cliente_asignado_a_instalacion: list[int]
    capacidad_restante_por_instalacion: list[int]
    iteraciones_con_mejora: int

### Componentes de GVNS

Seis funciones en secuencia: `sacudir_solucion` (perturbacion);
`_mejorar_con_1_shift` (N1, primera mejora, especifico de GVNS);
`_cerrar_instalaciones_poco_rentables` (N2), `_abrir_instalaciones_rentables`
(N3) e `_intercambiar_clientes_en_barrido` (N4) -- definidas en la seccion 1,
Busqueda local, y reutilizadas aqui tal cual --; coordinadas por
`descenso_por_vecindarios_variables` (VND).

#### Shaking

Reasigna k clientes al azar a instalaciones factibles alternativas.
El GVNS incrementa k cuando VND no mejora y lo reinicia a 1 cuando sí.

In [398]:
def sacudir_solucion(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    demanda_cliente: list[int],
    k: int,
    incompatibles_cliente: list[set[int]] | None = None,
) -> SolucionCFLP:
    instalacion_esta_abierta = list(instalacion_esta_abierta)
    instalacion_asignada_al_cliente = list(instalacion_asignada_al_cliente)
    capacidad_restante_instalacion = list(capacidad_restante_instalacion)

    clientes_a_perturbar = random.sample(
        range(numero_clientes), min(k, numero_clientes)
    )

    for indice_cliente in clientes_a_perturbar:
        demanda = demanda_cliente[indice_cliente]
        instalacion_origen = instalacion_asignada_al_cliente[indice_cliente]
        inc = incompatibles_cliente[indice_cliente] if incompatibles_cliente else set()

        candidatos_destino = [
            i for i in range(numero_instalaciones)
            if i != instalacion_origen
            and capacidad_restante_instalacion[i] >= demanda
            and not any(instalacion_asignada_al_cliente[k] == i for k in inc)
        ]

        if not candidatos_destino:
            continue

        instalacion_destino = random.choice(candidatos_destino)

        capacidad_restante_instalacion[instalacion_origen] += demanda
        capacidad_restante_instalacion[instalacion_destino] -= demanda
        instalacion_asignada_al_cliente[indice_cliente] = instalacion_destino
        instalacion_esta_abierta[instalacion_destino] = True

        sigue_ocupada = any(
            instalacion_asignada_al_cliente[j] == instalacion_origen
            for j in range(numero_clientes)
        )
        if not sigue_ocupada:
            instalacion_esta_abierta[instalacion_origen] = False

    return (
        instalacion_esta_abierta,
        instalacion_asignada_al_cliente,
        capacidad_restante_instalacion,
    )

#### N1 en GVNS: primera mejora

GVNS necesita una variante de N1 distinta a la de GRASP: en vez de barrer
todos los clientes reasignando cada mejora que encuentra
(`_reasignar_clientes_en_barrido`, seccion 1), se detiene en el primer
1-shift que reduzca el coste y devuelve el control de inmediato. Esta
politica de "primera mejora, un unico movimiento" es la que necesita el VND
para poder reiniciar desde N1 despues de cada paso y escalar a N2/N3/N4 solo
cuando N1 ya no encuentra nada -- con el barrido completo de GRASP, cada
llamada a N1 haria de por si varias reasignaciones antes de devolver el
control, cambiando como VND intercala los cuatro vecindarios. El delta que
usa es el mismo `_delta_reasignar_cliente` de la seccion 1; solo cambia
cuando parar.

In [399]:
def _mejorar_con_1_shift(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]] | None = None,
) -> tuple[SolucionCFLP, bool]:
    instalacion_esta_abierta = list(instalacion_esta_abierta)
    instalacion_asignada_al_cliente = list(instalacion_asignada_al_cliente)
    capacidad_restante_instalacion = list(capacidad_restante_instalacion)

    carga_por_instalacion = [0] * numero_instalaciones
    for j in range(numero_clientes):
        carga_por_instalacion[instalacion_asignada_al_cliente[j]] += 1

    for indice_cliente in range(numero_clientes):
        instalacion_actual = instalacion_asignada_al_cliente[indice_cliente]
        demanda = demanda_cliente[indice_cliente]
        inc = incompatibles_cliente[indice_cliente] if incompatibles_cliente else set()
        instalaciones_prohibidas = {instalacion_asignada_al_cliente[k] for k in inc}

        es_el_unico_en_origen = carga_por_instalacion[instalacion_actual] == 1

        for instalacion_destino in range(numero_instalaciones):
            if instalacion_destino == instalacion_actual:
                continue
            if capacidad_restante_instalacion[instalacion_destino] < demanda:
                continue
            if instalacion_destino in instalaciones_prohibidas:
                continue

            variacion_de_costo = _delta_reasignar_cliente(
                indice_cliente, instalacion_actual, instalacion_destino,
                instalacion_esta_abierta, costo_fijo_instalacion, demanda_cliente,
                costo_envio, es_el_unico_en_origen,
            )

            if variacion_de_costo < 0:
                capacidad_restante_instalacion[instalacion_actual] += demanda
                capacidad_restante_instalacion[instalacion_destino] -= demanda
                instalacion_asignada_al_cliente[indice_cliente] = (
                    instalacion_destino
                )
                instalacion_esta_abierta[instalacion_destino] = True
                if es_el_unico_en_origen:
                    instalacion_esta_abierta[instalacion_actual] = False
                return (
                    instalacion_esta_abierta,
                    instalacion_asignada_al_cliente,
                    capacidad_restante_instalacion,
                ), True

    return (
        instalacion_esta_abierta,
        instalacion_asignada_al_cliente,
        capacidad_restante_instalacion,
    ), False


#### VND: descenso por vecindarios variables

Prueba N1 (O(n*m)) antes de N2, N3 (cierre/apertura de instalaciones, mismo
orden de complejidad que N1 pero con una constante mayor) y N4 (intercambio
de dos clientes, seccion 1) y reinicia desde N1 tras cada mejora. Para cuando
ninguno de los cuatro produce mejora.

In [400]:
def descenso_por_vecindarios_variables(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    incompatibles_cliente: list[set[int]] | None = None,
) -> SolucionCFLP:
    solucion: SolucionCFLP = (
        list(instalacion_esta_abierta),
        list(instalacion_asignada_al_cliente),
        list(capacidad_restante_instalacion),
    )

    while True:
        solucion, mejoro_n1 = _mejorar_con_1_shift(
            solucion[0], solucion[1], solucion[2],
            numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio,
            incompatibles_cliente,
        )
        if mejoro_n1:
            continue

        solucion, mejoro_n2 = _cerrar_instalaciones_poco_rentables(
            solucion[0], solucion[1], solucion[2],
            numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio,
            incompatibles_cliente,
        )
        if mejoro_n2:
            continue

        solucion, mejoro_n3 = _abrir_instalaciones_rentables(
            solucion[0], solucion[1], solucion[2],
            numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio,
            incompatibles_cliente,
        )
        if mejoro_n3:
            continue

        solucion, mejoro_n4 = _intercambiar_clientes_en_barrido(
            solucion[0], solucion[1], solucion[2],
            numero_instalaciones, numero_clientes,
            costo_fijo_instalacion, demanda_cliente, costo_envio,
            incompatibles_cliente,
        )
        if mejoro_n4:
            continue

        break

    return solucion

### Ejecución de GVNS

In [401]:
def ejecutar_gvns(
    instalacion_esta_abierta: list[bool],
    instalacion_asignada_al_cliente: list[int],
    capacidad_restante_instalacion: list[int],
    numero_instalaciones: int,
    numero_clientes: int,
    capacidad_instalacion: list[int],
    costo_fijo_instalacion: list[int],
    demanda_cliente: list[int],
    costo_envio: list[list[int]],
    k_maximo: int = 3,
    numero_maximo_de_iteraciones: int = 30,
    semilla_aleatoria: int = 0,
    pares_incompatibles: list[tuple[int, int]] | None = None,
) -> ResultadoGVNS:
    random.seed(semilla_aleatoria)

    incompatibles = _precomputar_incompatibles(numero_clientes, pares_incompatibles or [])

    mejor_solucion: SolucionCFLP = (
        list(instalacion_esta_abierta),
        list(instalacion_asignada_al_cliente),
        list(capacidad_restante_instalacion),
    )
    mejor_costo = calcular_costo_total(
        mejor_solucion[0],
        mejor_solucion[1],
        costo_fijo_instalacion,
        demanda_cliente,
        costo_envio,
    )
    iteraciones_con_mejora = 0

    print("=" * 65)
    print("  GVNS - General Variable Neighborhood Search (MS-CFLP-CI)")
    print("=" * 65)
    print(f"  Instalaciones      : {numero_instalaciones}")
    print(f"  Clientes           : {numero_clientes}")
    print(f"  Pares incompatibles: {len(pares_incompatibles or [])}")
    print(f"  k máximo (shaking) : {k_maximo}")
    print(f"  Iteraciones máx.   : {numero_maximo_de_iteraciones}")
    print(f"  Costo inicial      : {mejor_costo:,.0f}")
    print("=" * 65)

    for iteracion in range(1, numero_maximo_de_iteraciones + 1):
        k = 1
        mejoro_en_esta_iteracion = False

        while k <= k_maximo:
            solucion_agitada = sacudir_solucion(
                mejor_solucion[0],
                mejor_solucion[1],
                mejor_solucion[2],
                numero_instalaciones,
                numero_clientes,
                demanda_cliente,
                k,
                incompatibles,
            )

            solucion_refinada = descenso_por_vecindarios_variables(
                solucion_agitada[0],
                solucion_agitada[1],
                solucion_agitada[2],
                numero_instalaciones,
                numero_clientes,
                costo_fijo_instalacion,
                demanda_cliente,
                costo_envio,
                incompatibles,
            )

            costo_refinado = calcular_costo_total(
                solucion_refinada[0],
                solucion_refinada[1],
                costo_fijo_instalacion,
                demanda_cliente,
                costo_envio,
            )

            if costo_refinado < mejor_costo:
                mejor_costo = costo_refinado
                mejor_solucion = (
                    list(solucion_refinada[0]),
                    list(solucion_refinada[1]),
                    list(solucion_refinada[2]),
                )
                k = 1
                mejoro_en_esta_iteracion = True
                iteraciones_con_mejora += 1
                print(
                    f"  Iter {iteracion:>3} | k={k} | "
                    f"Mejor: {mejor_costo:>10,.0f} << NUEVA MEJOR"
                )
            else:
                k += 1

        if not mejoro_en_esta_iteracion and (iteracion % 5 == 0):
            print(
                f"  Iter {iteracion:>3} | Sin mejora | "
                f"Mejor: {mejor_costo:>10,.0f}"
            )

    print("=" * 65)
    print(f"  SOLUCIÓN FINAL - Costo total: {mejor_costo:,.0f}")
    print(f"  Iteraciones con mejora: {iteraciones_con_mejora}")
    print("=" * 65)

    return {
        "costo_total_minimo": mejor_costo,
        "instalaciones_abiertas": list(mejor_solucion[0]),
        "cliente_asignado_a_instalacion": list(mejor_solucion[1]),
        "capacidad_restante_por_instalacion": list(mejor_solucion[2]),
        "iteraciones_con_mejora": iteraciones_con_mejora,
    }

### GVNS + Q-Learning: seleccion adaptativa de vecindarios

En el VND clasico el orden de aplicacion de vecindarios es **fijo**: primero se
agota N1 (1-shift), luego N2 (cierre de instalaciones), despues N3 (apertura
de instalaciones) y por ultimo N4 (intercambio de dos clientes); el descenso
reinicia desde N1 tras cualquier mejora y se declara minimo local cuando
ninguno de los cuatro mejora.

**GVNS+QL** sustituye ese orden fijo por una politica aprendida mediante
**Q-learning tabular**. En cada bloque del descenso, un agente decide que
vecindario *agotar* segun el estado actual (que vecindario se agoto en el bloque
anterior y si mejoro). La politica es epsilon-greedy con epsilon decreciente.
Ademas, la intensidad de sacudida `k` se elige mediante un bandido UCB1
independiente (`SelectorKUCB`) entre los valores aun no probados desde la
ultima mejora, en vez de seguir el ciclo fijo 1, 2, ..., k_max.

### Formulacion del agente Q-learning

**Vecindarios del portafolio** -- 0 = N1 (1-shift, reasignacion individual),
1 = N2 (cierre de instalaciones), 2 = N3 (apertura de instalaciones), 3 = N4
(intercambio de dos clientes).

**Accion = agotar un vecindario (bloque), no un movimiento suelto.** Cada accion
aplica el operador *first-improvement* (o, en el caso de N2/N3/N4, el barrido
completo) elegido en bucle hasta que deja de mejorar. El descenso termina, como
el VND, cuando los cuatro vecindarios quedan agotados sin mejora.

**Estados** -- capturan la dinamica reciente del descenso (9 en total: arranque
mas cuatro vecindarios por dos resultados posibles):

| Estado | Descripcion |
|--------|-------------|
| arranque | primer bloque del descenso (sin historial) |
| (N1, FALLO) | ultimo bloque fue N1 y no mejoro |
| (N1, OK) | ultimo bloque fue N1 y mejoro |
| (N2, FALLO) | ultimo bloque fue N2 y no mejoro |
| (N2, OK) | ultimo bloque fue N2 y mejoro |
| (N3, FALLO) | ultimo bloque fue N3 y no mejoro |
| (N3, OK) | ultimo bloque fue N3 y mejoro |
| (N4, FALLO) | ultimo bloque fue N4 y no mejoro |
| (N4, OK) | ultimo bloque fue N4 y mejoro |

**Enmascaramiento de acciones** -- un vecindario recien agotado sin mejora queda
*excluido* hasta que alguno de los vecindarios no excluidos logre una mejora,
momento en el que la exclusion se reinicia y los demas vuelven a estar
disponibles.

**Sesgo inicial (warm start)** -- la tabla Q se inicializa con un sesgo que
prefiere N1 > N2 > N3 > N4 en todas las filas, en vez de partir de cero.

**Recompensa** -- mejora relativa del bloque, normalizada por tiempo
(t_ref es una constante de referencia que evita divisiones por tiempos casi
nulos):

$$r_t = \begin{cases}
  \dfrac{C_{t-1} - C_t}{C_{t-1}} \cdot \dfrac{t_{\text{ref}}}{\Delta t + t_{\text{ref}}} & \text{si el bloque mejoro} \\[10pt]
  -0.001 \cdot \dfrac{\Delta t + t_{\text{ref}}}{t_{\text{ref}}} & \text{en caso contrario}
\end{cases}$$

**Actualizacion Q-learning** (regla TD off-policy):

$$Q(s,a) \;\leftarrow\; Q(s,a) + \alpha\!\left[\,r + \gamma\max_{a'}Q(s',a') - Q(s,a)\right]$$

**Parametros**: $\alpha = 0.2$, $\gamma = 0.8$, $\varepsilon_0 = 0.5$,
$\varepsilon_{\min} = 0.05$, $\varepsilon_{\text{decay}} = 0.999$. El agente usa un
generador de numeros aleatorios propio para la politica epsilon-greedy,
independiente del generador que usa el shaking.

In [402]:
import numpy as np

class SelectorQLearning:

    N_VEC = 4
    NOMBRES_VECINDARIOS = ("N1", "N2", "N3", "N4")
    TASA_ACTUALIZACION_T_REF = 0.1

    SESGO_INICIAL = [0.004, 0.003, 0.002, 0.001]

    def __init__(self, alpha: float = 0.2, gamma: float = 0.8,
                 epsilon: float = 0.5, epsilon_min: float = 0.05,
                 epsilon_decay: float = 0.999, semilla: int = 0,
                 warm_start: bool = True) -> None:
        self.alpha         = alpha
        self.gamma         = gamma
        self.epsilon       = epsilon
        self.epsilon_min   = epsilon_min
        self.epsilon_decay = epsilon_decay
        n_estados = self.N_VEC * 2 + 1
        if warm_start:
            sesgo = np.array(self.SESGO_INICIAL[: self.N_VEC])
            self.Q = np.tile(sesgo, (n_estados, 1))
        else:
            self.Q = np.zeros((n_estados, self.N_VEC))
        self.rng = random.Random(semilla)
        self.n_pasos = 0
        self.t_ref: float | None = None

    def codificar_estado(self, ultimo_vec: int | None, mejoro: bool) -> int:
        if ultimo_vec is None:
            return self.N_VEC * 2
        return ultimo_vec * 2 + (1 if mejoro else 0)

    def seleccionar_accion(self, estado: int, excluidas: frozenset = frozenset()) -> int:
        candidatas = [a for a in range(self.N_VEC) if a not in excluidas]
        if len(candidatas) == 1:
            return candidatas[0]
        if self.rng.random() < self.epsilon:
            return self.rng.choice(candidatas)
        return max(candidatas, key=lambda a: self.Q[estado, a])

    def actualizar_t_ref(self, duracion_bloque: float) -> None:
        if self.t_ref is None:
            self.t_ref = duracion_bloque
        else:
            self.t_ref = (
                (1 - self.TASA_ACTUALIZACION_T_REF) * self.t_ref
                + self.TASA_ACTUALIZACION_T_REF * duracion_bloque
            )

    def actualizar(self, estado: int, accion: int,
                   recompensa: float, estado_sig: int) -> None:
        mejor_futuro = float(np.max(self.Q[estado_sig]))
        td_error = recompensa + self.gamma * mejor_futuro - self.Q[estado, accion]
        self.Q[estado, accion] += self.alpha * td_error
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        self.n_pasos += 1

    def resumen(self) -> str:
        q = self.Q
        nombres = self.NOMBRES_VECINDARIOS

        def fila(indice_estado: int, etiqueta: str) -> str:
            valores = ",".join(f"{q[indice_estado, a]:.3f}" for a in range(self.N_VEC))
            mejor = nombres[int(np.argmax(q[indice_estado]))]
            return f"Q({etiqueta})=[{valores}]->{mejor}"

        arranque = self.N_VEC * 2
        t_ref_str = f"{self.t_ref*1000:.1f}ms" if self.t_ref is not None else "sin calibrar"
        partes = [f"ε={self.epsilon:.3f} | pasos={self.n_pasos} | t_ref={t_ref_str}",
                  fila(arranque, "arr")]
        for indice_vecindario, nombre in enumerate(nombres):
            partes.append(fila(indice_vecindario * 2, f"{nombre}✗"))
            partes.append(fila(indice_vecindario * 2 + 1, f"{nombre}✓"))
        return " | ".join(partes)

In [403]:
class SelectorKUCB:

    def __init__(self, k_maximo: int, c: float = 1.0) -> None:
        self.k_maximo = k_maximo
        self.c = c
        self.conteo = [0] * k_maximo
        self.suma_recompensa = [0.0] * k_maximo
        self.total_tiradas = 0
        self.ultimo_k: int | None = None

    def seleccionar_k(self, excluidos: frozenset = frozenset()) -> int:
        candidatos = [k for k in range(1, self.k_maximo + 1) if k not in excluidos]
        if len(candidatos) == 1:
            self.ultimo_k = candidatos[0]
            return candidatos[0]

        no_probados = [k for k in candidatos if self.conteo[k - 1] == 0]
        if no_probados:
            elegido = no_probados[0]
        else:
            ucb = [
                self.suma_recompensa[k - 1] / self.conteo[k - 1]
                + self.c * math.sqrt(2 * math.log(self.total_tiradas) / self.conteo[k - 1])
                for k in candidatos
            ]
            elegido = candidatos[max(range(len(candidatos)), key=lambda i: ucb[i])]
        self.ultimo_k = elegido
        return elegido

    def registrar(self, mejor_costo_antes: float, costo_resultante: float) -> None:
        recompensa = max(0.0, (mejor_costo_antes - costo_resultante) / mejor_costo_antes)
        k = self.ultimo_k
        self.conteo[k - 1] += 1
        self.suma_recompensa[k - 1] += recompensa
        self.total_tiradas += 1

    def resumen(self) -> str:
        partes = []
        for k in range(1, self.k_maximo + 1):
            n = self.conteo[k - 1]
            if n == 0:
                partes.append(f"k={k}:sin probar")
            else:
                media = self.suma_recompensa[k - 1] / n
                partes.append(f"k={k}:{media:.4f}({n})")
        return " | ".join(partes)


class AgentesGVNSQL:

    def __init__(self, agente_vecindario: SelectorQLearning, agente_k: SelectorKUCB) -> None:
        self.vecindario = agente_vecindario
        self.k = agente_k

    def resumen(self) -> str:
        return f"{self.vecindario.resumen()} || k-UCB1: {self.k.resumen()}"




In [404]:
import time as _t_ql

def seleccionar_solucion_aleatoria_del_pool(
    datos: DatosCFLP,
    tamano_pool: int = 5,
    semilla_aleatoria: int = 0,
) -> SolucionCFLP:
    random.seed(semilla_aleatoria)
    incompatibles = _precomputar_incompatibles(
        datos["numero_clientes"], datos["pares_incompatibles"]
    )
    pool = [
        construir_solucion_greedy_aleatoria(
            datos["numero_instalaciones"],
            datos["numero_clientes"],
            datos["capacidad_instalacion"],
            datos["costo_fijo_instalacion"],
            datos["demanda_cliente"],
            datos["costo_envio"],
            round(random.uniform(0.0, 1.0), 2),
            incompatibles,
        )
        for _ in range(tamano_pool)
    ]
    return random.choice(pool)


def _ejecutar_gvns_iter(
    datos: DatosCFLP,
    abierta: list[bool],
    asignada: list[int],
    restante: list[int],
    k_maximo: int,
    n_iter: int,
    semilla: int,
    max_mejoras: int | None = None,
) -> float:
    if max_mejoras is None:
        max_mejoras = n_iter * k_maximo
    random.seed(semilla)
    F    = datos["numero_instalaciones"]
    C    = datos["numero_clientes"]
    cap  = datos["capacidad_instalacion"]
    cf   = datos["costo_fijo_instalacion"]
    dem  = datos["demanda_cliente"]
    envio = datos["costo_envio"]
    incompatibles = _precomputar_incompatibles(C, datos["pares_incompatibles"])

    mejor_sol = (list(abierta), list(asignada), list(restante))
    mejor_costo = calcular_costo_total(mejor_sol[0], mejor_sol[1], cf, dem, envio)
    mejoras_total = 0

    for _ in range(n_iter):
        k = 1
        while k <= k_maximo and mejoras_total < max_mejoras:
            agitada = sacudir_solucion(
                mejor_sol[0], mejor_sol[1], mejor_sol[2], F, C, dem, k, incompatibles
            )
            refinada = mejorar_solucion_con_busqueda_local(
                agitada[0], agitada[1], agitada[2], F, C, cap, cf, dem, envio, incompatibles
            )
            costo_ref = calcular_costo_total(refinada[0], refinada[1], cf, dem, envio)
            if costo_ref < mejor_costo:
                mejor_costo   = costo_ref
                mejor_sol     = (list(refinada[0]), list(refinada[1]), list(refinada[2]))
                mejoras_total += 1
                k = 1
            else:
                k += 1

    return mejor_costo







def _ejecutar_gvns_iter_vnd(
    datos: DatosCFLP,
    abierta: list[bool],
    asignada: list[int],
    restante: list[int],
    k_maximo: int,
    n_iter: int,
    semilla: int,
    max_mejoras: int | None = None,
    tiempo_limite_s: float | None = None,
) -> float:
    if max_mejoras is None:
        max_mejoras = n_iter * k_maximo
    random.seed(semilla)
    t_inicio = time.perf_counter()
    F, C = datos["numero_instalaciones"], datos["numero_clientes"]
    cf   = datos["costo_fijo_instalacion"]
    dem  = datos["demanda_cliente"]
    envio = datos["costo_envio"]
    incompatibles = _precomputar_incompatibles(C, datos["pares_incompatibles"])

    mejor_sol   = (list(abierta), list(asignada), list(restante))
    mejor_costo = calcular_costo_total(mejor_sol[0], mejor_sol[1], cf, dem, envio)
    mejoras_total = 0

    for _ in range(n_iter):
        if mejoras_total >= max_mejoras:
            break
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break
        ya_probado: set[int] = set()
        while len(ya_probado) < k_maximo:
            if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
                break
            k = 1 if 1 not in ya_probado else min(
                j for j in range(1, k_maximo + 1) if j not in ya_probado
            )
            agitada  = sacudir_solucion(
                mejor_sol[0], mejor_sol[1], mejor_sol[2], F, C, dem, k, incompatibles
            )
            refinada = descenso_por_vecindarios_variables(
                agitada[0], agitada[1], agitada[2], F, C, cf, dem, envio, incompatibles
            )
            costo_ref = calcular_costo_total(refinada[0], refinada[1], cf, dem, envio)
            if costo_ref < mejor_costo:
                mejor_costo   = costo_ref
                mejor_sol     = refinada
                mejoras_total += 1
                ya_probado.clear()
                if mejoras_total >= max_mejoras:
                    break
            else:
                ya_probado.add(k)
    return mejor_costo



def _agotar_vecindario(operador, sol, F, C, cf, dem, envio, incompatibles):
    mejoro_alguna = False
    while True:
        sol, mejoro = operador(sol[0], sol[1], sol[2],
                               F, C, cf, dem, envio, incompatibles)
        if not mejoro:
            return sol, mejoro_alguna
        mejoro_alguna = True


def _intercambiar_en_barrido_gvns(
    instalacion_esta_abierta, instalacion_asignada_al_cliente,
    capacidad_restante_instalacion, numero_instalaciones, numero_clientes,
    costo_fijo_instalacion, demanda_cliente, costo_envio,
    incompatibles_cliente=None,
):
    return _intercambiar_clientes_en_barrido(
        instalacion_esta_abierta, instalacion_asignada_al_cliente,
        capacidad_restante_instalacion, numero_instalaciones, numero_clientes,
        costo_fijo_instalacion, demanda_cliente, costo_envio,
        incompatibles_cliente,
    )


def _ejecutar_gvns_iter_ql(
    datos: DatosCFLP,
    abierta: list[bool],
    asignada: list[int],
    restante: list[int],
    k_maximo: int,
    n_iter: int,
    semilla: int,
    max_mejoras: int | None = None,
    tiempo_limite_s: float | None = None,
) -> tuple[float, SelectorQLearning, "SelectorKUCB"]:
    if max_mejoras is None:
        max_mejoras = n_iter * k_maximo
    random.seed(semilla)
    t_inicio = time.perf_counter()
    F, C = datos["numero_instalaciones"], datos["numero_clientes"]
    cf    = datos["costo_fijo_instalacion"]
    dem   = datos["demanda_cliente"]
    envio = datos["costo_envio"]
    incompatibles = _precomputar_incompatibles(C, datos["pares_incompatibles"])
    operadores = [_mejorar_con_1_shift, _cerrar_instalaciones_poco_rentables, _abrir_instalaciones_rentables, _intercambiar_en_barrido_gvns]

    agente      = SelectorQLearning(semilla=semilla + 7919)
    agente_k    = SelectorKUCB(k_maximo)
    mejor_sol   = (list(abierta), list(asignada), list(restante))
    mejor_costo = calcular_costo_total(mejor_sol[0], mejor_sol[1], cf, dem, envio)
    mejoras_total = 0

    for _ in range(n_iter):
        if mejoras_total >= max_mejoras:
            break
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break
        ya_probado: set[int] = set()

        while len(ya_probado) < k_maximo:
            if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
                break
            k = agente_k.seleccionar_k(excluidos=frozenset(ya_probado))
            costo_antes_shake = mejor_costo
            sol_actual   = sacudir_solucion(
                mejor_sol[0], mejor_sol[1], mejor_sol[2], F, C, dem, k, incompatibles
            )
            costo_actual = calcular_costo_total(sol_actual[0], sol_actual[1], cf, dem, envio)

            ultimo_vec: int | None = None
            mejoro_bloque          = False
            agotados: set[int]     = set()

            while len(agotados) < agente.N_VEC:
                estado = agente.codificar_estado(ultimo_vec, mejoro_bloque)
                accion = agente.seleccionar_accion(estado, excluidas=frozenset(agotados))

                t0 = _t_ql.perf_counter()
                sol_nueva, mejoro_bloque = _agotar_vecindario(
                    operadores[accion], sol_actual, F, C, cf, dem, envio, incompatibles
                )
                dt = _t_ql.perf_counter() - t0
                t_ref_actual = agente.t_ref if agente.t_ref is not None else dt

                if mejoro_bloque:
                    nuevo_costo = calcular_costo_total(sol_nueva[0], sol_nueva[1], cf, dem, envio)
                    recompensa  = ((costo_actual - nuevo_costo) / costo_actual) * t_ref_actual / (dt + t_ref_actual)
                    sol_actual, costo_actual = sol_nueva, nuevo_costo
                    agotados = {accion}
                else:
                    recompensa = -0.001 * (dt + t_ref_actual) / t_ref_actual
                    agotados.add(accion)

                agente.actualizar_t_ref(dt)
                estado_sig = agente.codificar_estado(accion, mejoro_bloque)
                agente.actualizar(estado, accion, recompensa, estado_sig)
                ultimo_vec = accion

            agente_k.registrar(costo_antes_shake, costo_actual)

            if costo_actual < mejor_costo:
                mejor_costo   = costo_actual
                mejor_sol     = sol_actual
                mejoras_total += 1
                ya_probado.clear()
                if mejoras_total >= max_mejoras:
                    break
            else:
                ya_probado.add(k)

    return mejor_costo, agente, agente_k

### Benchmark GVNS+VND (base) vs GVNS+QL

Compara **GVNS+VND** de orden fijo (N1->N2->N3->N4) contra **GVNS+QL** (mismo
portafolio de cuatro vecindarios, orden aprendido, mas intensidad de sacudida
elegida por un bandido UCB1) sobre la misma solucion inicial y la misma
secuencia de shaking por semilla. Se reportan tres magnitudes: *gap* de
calidad respecto a la referencia, tiempo, y el speedup de QL sobre la base.

In [405]:
import os as _os_gb
import time as _tgb
import numpy as np

_EJECUTAR_BENCHMARK_GVNS_QL = False

if _EJECUTAR_BENCHMARK_GVNS_QL:

    _INST_BENCH     = ["wlp01.dzn", "wlp02.dzn", "wlp03.dzn", "wlp04.dzn",
                        "wlp05.dzn", "wlp06.dzn", "wlp07.dzn", "wlp08.dzn"]
    _REF_BENCH      = {"wlp01": 28716, "wlp02": 52952, "wlp03": 64296, "wlp04": 84633,
                        "wlp05": 103857, "wlp06": 111654, "wlp07": 162277, "wlp08": 187938}
    _SEMILLAS_BENCH = [0, 1, 2, 3, 4]
    _K_MAX_BENCH    = 10
    _N_ITER_BENCH   = 30
    _POOL_BENCH     = 10

    _gb_res = {
        nombre: {"GVNS": {"gaps": [], "t": []},
                 "GVNS+QL": {"gaps": [], "t": []}}
        for nombre in [a.replace(".dzn", "") for a in _INST_BENCH]
    }

    for archivo in _INST_BENCH:
        nombre = archivo.replace(".dzn", "")
        datos  = leer_instancia_dzn(_os_gb.path.join("Instaces_MS-CFLP-CI/Public", archivo))
        ref    = _REF_BENCH[nombre]
        incompatibles = _precomputar_incompatibles(
            datos["numero_clientes"], datos["pares_incompatibles"]
        )

        print(f"\n  {nombre}")
        for semilla in _SEMILLAS_BENCH:
            ab0, asig0, rest0 = seleccionar_solucion_aleatoria_del_pool(datos, _POOL_BENCH, semilla)
            ab0, asig0, rest0 = mejorar_solucion_con_busqueda_local(
                ab0, asig0, rest0,
                datos["numero_instalaciones"], datos["numero_clientes"],
                datos["capacidad_instalacion"], datos["costo_fijo_instalacion"],
                datos["demanda_cliente"], datos["costo_envio"], incompatibles,
            )

            t0 = _tgb.perf_counter()
            costo_base = _ejecutar_gvns_iter_vnd(
                datos, list(ab0), list(asig0), list(rest0),
                _K_MAX_BENCH, _N_ITER_BENCH, semilla,
            )
            t_base   = _tgb.perf_counter() - t0
            gap_base = 100 * (costo_base - ref) / ref
            _gb_res[nombre]["GVNS"]["gaps"].append(gap_base)
            _gb_res[nombre]["GVNS"]["t"].append(t_base)

            t0 = _tgb.perf_counter()
            costo_ql, agente, agente_k = _ejecutar_gvns_iter_ql(
                datos, list(ab0), list(asig0), list(rest0),
                _K_MAX_BENCH, _N_ITER_BENCH, semilla,
            )
            t_ql   = _tgb.perf_counter() - t0
            gap_ql = 100 * (costo_ql - ref) / ref
            _gb_res[nombre]["GVNS+QL"]["gaps"].append(gap_ql)
            _gb_res[nombre]["GVNS+QL"]["t"].append(t_ql)

            speedup = 100 * (t_base - t_ql) / t_base
            print(f"    seed={semilla}  GVNS: {gap_base:+.2f}% {t_base:.1f}s  "
                  f"QL: {gap_ql:+.2f}% {t_ql:.1f}s  Δgap={gap_ql-gap_base:+.2f}pp  "
                  f"speedup={speedup:+.0f}%")

    instancias = [a.replace(".dzn", "") for a in _INST_BENCH]
    print(f"\n{'='*76}")
    print(f"  {'':<10}  {'── GVNS base ──':^22}  {'── GVNS+QL ──':^22}  {'Δgap':>6}  {'spd':>5}")
    print(f"  {'Inst.':<10}  {'gap%':>7}  {'σ':>5}  {'t(s)':>5}  "
          f"{'gap%':>7}  {'σ':>5}  {'t(s)':>5}  {'(pp)':>6}  {'(%)':>5}")
    print(f"  {'-'*76}")

    all_base, all_ql = [], []
    for nombre in instancias:
        bg = np.mean(_gb_res[nombre]["GVNS"]["gaps"])
        bs = np.std(_gb_res[nombre]["GVNS"]["gaps"])
        bt = np.mean(_gb_res[nombre]["GVNS"]["t"])
        qg = np.mean(_gb_res[nombre]["GVNS+QL"]["gaps"])
        qs = np.std(_gb_res[nombre]["GVNS+QL"]["gaps"])
        qt = np.mean(_gb_res[nombre]["GVNS+QL"]["t"])
        spd = 100 * (bt - qt) / bt
        all_base.append(bg); all_ql.append(qg)
        print(f"  {nombre:<10}  {bg:>+7.2f}  {bs:>5.2f}  {bt:>5.1f}  "
              f"{qg:>+7.2f}  {qs:>5.2f}  {qt:>5.1f}  {qg-bg:>+6.2f}  {spd:>+5.0f}")

    print(f"  {'-'*76}")
    mb = np.mean(all_base); mq = np.mean(all_ql)
    print(f"  {'Media':<10}  {mb:>+7.2f}  {'':<5}  {'':<5}  {mq:>+7.2f}  {'':<5}  {'':<5}  {mq-mb:>+6.2f}")
    print(f"  {'='*76}")
    print(f"\n  Lectura: la calidad (gap) de GVNS+QL iguala a la base dentro del ruido")
    print(f"  (Δgap ≈ 0), mientras que el speedup positivo refleja el ahorro de las")
    print(f"  pasadas de N2/N3 que el agente aprende a omitir. Política aprendida:")
    print(f"    {agente.resumen()}")
    print(f"    k-UCB1: {agente_k.resumen()}")
else:
    print("Benchmark GVNS+VND vs GVNS+QL omitido (_EJECUTAR_BENCHMARK_GVNS_QL = False)")


Benchmark GVNS+VND vs GVNS+QL omitido (_EJECUTAR_BENCHMARK_GVNS_QL = False)


## 4. Enfriamiento Simulado + ML

SA acepta movimientos de empeoramiento con probabilidad e^{-Delta/T}, donde la temperatura T
decrece multiplicando por un ratio de enfriamiento en cada bloque de movimientos.

`SelectorRatioUCB` (mas abajo) elige el ratio de enfriamiento en tiempo de
ejecucion mediante un bandido UCB1, en vez de dejarlo fijo.

### Algoritmo base

Construye la solucion inicial con GRASP (alfa = 0.3) seguido de busqueda local. En cada
iteracion del bucle principal se sortea uno de los cuatro tipos de movimiento -- N1
(reasignar un cliente), N2 (cerrar una instalacion abierta), N3 (abrir una instalacion
cerrada) o N4 (intercambiar dos clientes) -- y se le aplica el criterio de Metropolis.
Con un unico cliente por movimiento (solo N1), SA nunca podria reconsiderar que
instalaciones estan abiertas salvo por efecto colateral; N2, N3 y N4 le dan la misma
capacidad de reestructurar la solucion que ya tiene la busqueda local que lo inicializa,
con la diferencia de que aqui una reestructuracion perjudicial tambien puede aceptarse
mientras la temperatura lo permita.

In [406]:
def recocido_simulado(
    datos: DatosCFLP,
    n_movimientos: int = 5_000,
    ratio_inicial: float = 0.995,
    semilla: int = 0,
    tiempo_limite_s: float | None = None,
) -> ResultadoBenchmark:
    random.seed(semilla)
    t_inicio    = time.perf_counter()
    num_instalaciones = datos["numero_instalaciones"]
    num_clientes      = datos["numero_clientes"]
    capacidad         = datos["capacidad_instalacion"]
    costo_fijo        = datos["costo_fijo_instalacion"]
    demanda           = datos["demanda_cliente"]
    envio             = datos["costo_envio"]
    incompatibles     = _precomputar_incompatibles(num_clientes, datos["pares_incompatibles"])

    abierta, asignada, restante = construir_solucion_greedy_aleatoria(
        num_instalaciones, num_clientes, capacidad, costo_fijo, demanda, envio, 0.3,
        incompatibles,
    )
    abierta, asignada, restante = mejorar_solucion_con_busqueda_local(
        abierta, asignada, restante,
        num_instalaciones, num_clientes, capacidad, costo_fijo, demanda, envio,
        incompatibles,
    )
    costo = calcular_costo_total(abierta, asignada, costo_fijo, demanda, envio)

    mejor_abierta  = list(abierta)
    mejor_asignada = list(asignada)
    mejor_costo    = costo

    clientes_por_instalacion = [set() for _ in range(num_instalaciones)]
    for cliente in range(num_clientes):
        clientes_por_instalacion[asignada[cliente]].add(cliente)

    temperatura = costo * 0.02
    ratio       = ratio_inicial

    for num_iter in range(n_movimientos):
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break

        tipo_movimiento = random.choice(("N1", "N2", "N3", "N4"))

        if tipo_movimiento == "N1":
            cliente            = random.randint(0, num_clientes - 1)
            instalacion_origen = asignada[cliente]
            demanda_cliente    = demanda[cliente]
            inc                = incompatibles[cliente]

            instalaciones_prohibidas = {asignada[k] for k in inc}
            candidatas = [
                inst for inst in range(num_instalaciones)
                if inst != instalacion_origen
                and restante[inst] >= demanda_cliente
                and inst not in instalaciones_prohibidas
            ]
            if not candidatas:
                continue
            instalacion_destino = random.choice(candidatas)

            abre_destino  = not abierta[instalacion_destino]
            cierra_origen = len(clientes_por_instalacion[instalacion_origen]) == 1
            delta = _delta_reasignar_cliente(
                cliente, instalacion_origen, instalacion_destino,
                abierta, costo_fijo, demanda, envio, cierra_origen,
            )

            acepta = delta <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta / temperatura))
            if acepta:
                restante[instalacion_origen]  += demanda_cliente
                restante[instalacion_destino] -= demanda_cliente
                clientes_por_instalacion[instalacion_origen].discard(cliente)
                clientes_por_instalacion[instalacion_destino].add(cliente)
                asignada[cliente] = instalacion_destino
                if abre_destino:
                    abierta[instalacion_destino] = True
                if cierra_origen:
                    abierta[instalacion_origen] = False
                costo += delta
                if costo < mejor_costo:
                    mejor_costo    = costo
                    mejor_abierta  = list(abierta)
                    mejor_asignada = list(asignada)

        elif tipo_movimiento == "N4":
            cliente_a = random.randint(0, num_clientes - 1)
            instalacion_a = asignada[cliente_a]
            demanda_a = demanda[cliente_a]
            candidatos = []
            for cliente_b in range(num_clientes):
                instalacion_b = asignada[cliente_b]
                if instalacion_b == instalacion_a:
                    continue
                demanda_b = demanda[cliente_b]
                if restante[instalacion_a] + demanda_a < demanda_b:
                    continue
                if restante[instalacion_b] + demanda_b < demanda_a:
                    continue
                if any(k != cliente_b and asignada[k] == instalacion_b for k in incompatibles[cliente_a]):
                    continue
                if any(k != cliente_a and asignada[k] == instalacion_a for k in incompatibles[cliente_b]):
                    continue
                candidatos.append(cliente_b)
            if not candidatos:
                continue
            cliente_b = random.choice(candidatos)
            instalacion_b = asignada[cliente_b]
            demanda_b = demanda[cliente_b]

            delta = _delta_intercambiar_clientes(
                cliente_a, cliente_b, instalacion_a, instalacion_b,
                demanda, envio,
            )
            acepta = delta <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta / temperatura))
            if acepta:
                asignada[cliente_a] = instalacion_b
                asignada[cliente_b] = instalacion_a
                restante[instalacion_a] += demanda_a - demanda_b
                restante[instalacion_b] += demanda_b - demanda_a
                clientes_por_instalacion[instalacion_a].discard(cliente_a)
                clientes_por_instalacion[instalacion_a].add(cliente_b)
                clientes_por_instalacion[instalacion_b].discard(cliente_b)
                clientes_por_instalacion[instalacion_b].add(cliente_a)
                costo += delta
                if costo < mejor_costo:
                    mejor_costo    = costo
                    mejor_abierta  = list(abierta)
                    mejor_asignada = list(asignada)

        elif tipo_movimiento == "N2":
            abiertas = [i for i in range(num_instalaciones) if abierta[i]]
            if len(abiertas) <= 1:
                continue
            instalacion = random.choice(abiertas)
            propuesta = _evaluar_cierre_instalacion(
                instalacion, abierta, asignada, restante,
                num_instalaciones, num_clientes, costo_fijo, demanda, envio, incompatibles,
            )
            if propuesta is None:
                continue
            delta, reasignacion = propuesta
            delta_aceptacion = delta / len(reasignacion)
            acepta = delta_aceptacion <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta_aceptacion / temperatura))
            if acepta:
                for cliente, nueva_instalacion in reasignacion.items():
                    d = demanda[cliente]
                    restante[instalacion]      += d
                    restante[nueva_instalacion] -= d
                    clientes_por_instalacion[instalacion].discard(cliente)
                    clientes_por_instalacion[nueva_instalacion].add(cliente)
                    asignada[cliente] = nueva_instalacion
                    abierta[nueva_instalacion] = True
                abierta[instalacion] = False
                costo += delta
                if costo < mejor_costo:
                    mejor_costo    = costo
                    mejor_abierta  = list(abierta)
                    mejor_asignada = list(asignada)

        else:
            cerradas = [i for i in range(num_instalaciones) if not abierta[i]]
            if not cerradas:
                continue
            instalacion = random.choice(cerradas)
            propuesta = _evaluar_apertura_instalacion(
                instalacion, abierta, asignada, restante,
                num_clientes, costo_fijo, demanda, envio, incompatibles,
                [len(s) for s in clientes_por_instalacion],
            )
            if propuesta is None:
                continue
            delta, seleccionados = propuesta
            delta_aceptacion = delta / len(seleccionados)
            acepta = delta_aceptacion <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta_aceptacion / temperatura))
            if acepta:
                origenes_afectados = set()
                for cliente in seleccionados:
                    origen = asignada[cliente]
                    d = demanda[cliente]
                    restante[origen]      += d
                    restante[instalacion] -= d
                    clientes_por_instalacion[origen].discard(cliente)
                    clientes_por_instalacion[instalacion].add(cliente)
                    asignada[cliente] = instalacion
                    origenes_afectados.add(origen)
                abierta[instalacion] = True
                for origen in origenes_afectados:
                    if not clientes_por_instalacion[origen]:
                        abierta[origen] = False
                costo += delta
                if costo < mejor_costo:
                    mejor_costo    = costo
                    mejor_abierta  = list(abierta)
                    mejor_asignada = list(asignada)

        if (num_iter + 1) % 200 == 0:
            temperatura *= ratio
            if temperatura < 1e-10:
                temperatura = mejor_costo * 5e-4

    return ResultadoBenchmark(
        nombre="SA",
        costo_final=mejor_costo,
        tiempo_total_s=time.perf_counter() - t_inicio,
    )

### Componente ML: SelectorRatioUCB

El SA base (celda anterior) usa un ratio de enfriamiento **fijo**
(`ratio_inicial`). `SelectorRatioUCB` deja que un bandido multi-brazo **UCB1**
decida el ratio sobre la marcha: cada brazo es uno de los ratios candidatos
`{0.90, 0.95, 0.99, 0.995, 0.999}`. La recompensa de un intervalo es su mejora
absoluta de coste, normalizada min-max contra el historial de mejoras
observadas hasta el momento (1 = la mejor mejora vista hasta ahora, 0 = sin
mejora).

In [407]:
class SelectorRatioUCB:

    RATIOS = [0.90, 0.95, 0.99, 0.995, 0.999]

    def __init__(self, c: float = 1.0) -> None:
        self.c                       = c
        self.conteo                  = [0] * len(self.RATIOS)
        self.suma_recompensa         = [0.0] * len(self.RATIOS)
        self.total_tiradas           = 0
        self.ultimo_brazo: int | None = None
        self.historial_mejoras: list[float] = []
        self.tiempo_entrenamiento_s = 0.0
        self.tiempo_inferencia_s    = 0.0

    def seleccionar_ratio(self) -> float:
        t0 = time.perf_counter()
        no_probados = [i for i, n in enumerate(self.conteo) if n == 0]
        if no_probados:
            indice = no_probados[0]
        else:
            ucb = [
                self.suma_recompensa[i] / self.conteo[i]
                + self.c * math.sqrt(2 * math.log(self.total_tiradas) / self.conteo[i])
                for i in range(len(self.RATIOS))
            ]
            indice = max(range(len(self.RATIOS)), key=lambda i: ucb[i])
        self.ultimo_brazo = indice
        self.tiempo_inferencia_s += time.perf_counter() - t0
        return self.RATIOS[indice]

    def registrar_recompensa(self, mejora_absoluta: float) -> None:
        self.historial_mejoras.append(mejora_absoluta)
        mmax = max(self.historial_mejoras)
        recompensa = mejora_absoluta / mmax if mmax > 0 else 0.0

        indice = self.ultimo_brazo
        self.conteo[indice]          += 1
        self.suma_recompensa[indice] += recompensa
        self.total_tiradas           += 1


### SA con control adaptativo de temperatura

`recocido_simulado_ml` es identico al SA base -- mismos cuatro movimientos
N1/N2/N3/N4, mismo criterio de Metropolis -- salvo que el ratio de enfriamiento
no es fijo: cada `intervalo_decidir` movimientos, `SelectorRatioUCB` elige uno
nuevo segun las recompensas acumuladas por cada ratio.

In [408]:
def recocido_simulado_ml(
    datos: DatosCFLP,
    n_movimientos: int = 5_000,
    intervalo_enfriar: int = 200,
    intervalo_decidir: int = 2_000,
    semilla: int = 0,
    tiempo_limite_s: float | None = None,
) -> ResultadoBenchmark:
    random.seed(semilla)
    t_inicio    = time.perf_counter()
    num_instalaciones = datos["numero_instalaciones"]
    num_clientes      = datos["numero_clientes"]
    capacidad         = datos["capacidad_instalacion"]
    costo_fijo        = datos["costo_fijo_instalacion"]
    demanda           = datos["demanda_cliente"]
    envio             = datos["costo_envio"]
    incompatibles     = _precomputar_incompatibles(num_clientes, datos["pares_incompatibles"])

    abierta, asignada, restante = construir_solucion_greedy_aleatoria(
        num_instalaciones, num_clientes, capacidad, costo_fijo, demanda, envio, 0.3,
        incompatibles,
    )
    abierta, asignada, restante = mejorar_solucion_con_busqueda_local(
        abierta, asignada, restante,
        num_instalaciones, num_clientes, capacidad, costo_fijo, demanda, envio,
        incompatibles,
    )
    costo = calcular_costo_total(abierta, asignada, costo_fijo, demanda, envio)

    mejor_abierta  = list(abierta)
    mejor_asignada = list(asignada)
    mejor_costo    = costo

    clientes_por_instalacion = [set() for _ in range(num_instalaciones)]
    for cliente in range(num_clientes):
        clientes_por_instalacion[asignada[cliente]].add(cliente)

    temperatura = costo * 0.02
    selector    = SelectorRatioUCB()
    ratio       = selector.seleccionar_ratio()
    mejor_costo_intervalo = mejor_costo

    num_iter = 0
    while num_iter < n_movimientos:
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break

        tipo_movimiento = random.choice(("N1", "N2", "N3", "N4"))

        if tipo_movimiento == "N1":
            cliente            = random.randint(0, num_clientes - 1)
            instalacion_origen = asignada[cliente]
            demanda_cliente    = demanda[cliente]
            inc                = incompatibles[cliente]

            instalaciones_prohibidas = {asignada[k] for k in inc}
            candidatas = [
                inst for inst in range(num_instalaciones)
                if inst != instalacion_origen
                and restante[inst] >= demanda_cliente
                and inst not in instalaciones_prohibidas
            ]
            if candidatas:
                instalacion_destino = random.choice(candidatas)

                abre_destino  = not abierta[instalacion_destino]
                cierra_origen = len(clientes_por_instalacion[instalacion_origen]) == 1
                delta = _delta_reasignar_cliente(
                    cliente, instalacion_origen, instalacion_destino,
                    abierta, costo_fijo, demanda, envio, cierra_origen,
                )

                acepta = delta <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta / temperatura))
                if acepta:
                    restante[instalacion_origen]  += demanda_cliente
                    restante[instalacion_destino] -= demanda_cliente
                    clientes_por_instalacion[instalacion_origen].discard(cliente)
                    clientes_por_instalacion[instalacion_destino].add(cliente)
                    asignada[cliente] = instalacion_destino
                    if abre_destino:
                        abierta[instalacion_destino] = True
                    if cierra_origen:
                        abierta[instalacion_origen] = False
                    costo += delta
                    if costo < mejor_costo:
                        mejor_costo    = costo
                        mejor_abierta  = list(abierta)
                        mejor_asignada = list(asignada)

        elif tipo_movimiento == "N4":
            cliente_a = random.randint(0, num_clientes - 1)
            instalacion_a = asignada[cliente_a]
            demanda_a = demanda[cliente_a]
            candidatos = []
            for cliente_b in range(num_clientes):
                instalacion_b = asignada[cliente_b]
                if instalacion_b == instalacion_a:
                    continue
                demanda_b = demanda[cliente_b]
                if restante[instalacion_a] + demanda_a < demanda_b:
                    continue
                if restante[instalacion_b] + demanda_b < demanda_a:
                    continue
                if any(k != cliente_b and asignada[k] == instalacion_b for k in incompatibles[cliente_a]):
                    continue
                if any(k != cliente_a and asignada[k] == instalacion_a for k in incompatibles[cliente_b]):
                    continue
                candidatos.append(cliente_b)
            if not candidatos:
                continue
            cliente_b = random.choice(candidatos)
            instalacion_b = asignada[cliente_b]
            demanda_b = demanda[cliente_b]

            delta = _delta_intercambiar_clientes(
                cliente_a, cliente_b, instalacion_a, instalacion_b,
                demanda, envio,
            )
            acepta = delta <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta / temperatura))
            if acepta:
                asignada[cliente_a] = instalacion_b
                asignada[cliente_b] = instalacion_a
                restante[instalacion_a] += demanda_a - demanda_b
                restante[instalacion_b] += demanda_b - demanda_a
                clientes_por_instalacion[instalacion_a].discard(cliente_a)
                clientes_por_instalacion[instalacion_a].add(cliente_b)
                clientes_por_instalacion[instalacion_b].discard(cliente_b)
                clientes_por_instalacion[instalacion_b].add(cliente_a)
                costo += delta
                if costo < mejor_costo:
                    mejor_costo    = costo
                    mejor_abierta  = list(abierta)
                    mejor_asignada = list(asignada)

        elif tipo_movimiento == "N2":
            abiertas = [i for i in range(num_instalaciones) if abierta[i]]
            if len(abiertas) > 1:
                instalacion = random.choice(abiertas)
                propuesta = _evaluar_cierre_instalacion(
                    instalacion, abierta, asignada, restante,
                    num_instalaciones, num_clientes, costo_fijo, demanda, envio, incompatibles,
                )
                if propuesta is not None:
                    delta, reasignacion = propuesta
                    delta_aceptacion = delta / len(reasignacion)
                    acepta = delta_aceptacion <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta_aceptacion / temperatura))
                    if acepta:
                        for cliente, nueva_instalacion in reasignacion.items():
                            d = demanda[cliente]
                            restante[instalacion]      += d
                            restante[nueva_instalacion] -= d
                            clientes_por_instalacion[instalacion].discard(cliente)
                            clientes_por_instalacion[nueva_instalacion].add(cliente)
                            asignada[cliente] = nueva_instalacion
                            abierta[nueva_instalacion] = True
                        abierta[instalacion] = False
                        costo += delta
                        if costo < mejor_costo:
                            mejor_costo    = costo
                            mejor_abierta  = list(abierta)
                            mejor_asignada = list(asignada)

        else:
            cerradas = [i for i in range(num_instalaciones) if not abierta[i]]
            if cerradas:
                instalacion = random.choice(cerradas)
                propuesta = _evaluar_apertura_instalacion(
                    instalacion, abierta, asignada, restante,
                    num_clientes, costo_fijo, demanda, envio, incompatibles,
                    [len(s) for s in clientes_por_instalacion],
                )
                if propuesta is not None:
                    delta, seleccionados = propuesta
                    delta_aceptacion = delta / len(seleccionados)
                    acepta = delta_aceptacion <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta_aceptacion / temperatura))
                    if acepta:
                        origenes_afectados = set()
                        for cliente in seleccionados:
                            origen = asignada[cliente]
                            d = demanda[cliente]
                            restante[origen]      += d
                            restante[instalacion] -= d
                            clientes_por_instalacion[origen].discard(cliente)
                            clientes_por_instalacion[instalacion].add(cliente)
                            asignada[cliente] = instalacion
                            origenes_afectados.add(origen)
                        abierta[instalacion] = True
                        for origen in origenes_afectados:
                            if not clientes_por_instalacion[origen]:
                                abierta[origen] = False
                        costo += delta
                        if costo < mejor_costo:
                            mejor_costo    = costo
                            mejor_abierta  = list(abierta)
                            mejor_asignada = list(asignada)

        if (num_iter + 1) % intervalo_enfriar == 0:
            temperatura *= ratio
            if temperatura < 1e-10:
                temperatura = mejor_costo * 5e-4

        if (num_iter + 1) % intervalo_decidir == 0:
            if selector.ultimo_brazo is not None:
                mejora_absoluta = max(0.0, mejor_costo_intervalo - mejor_costo)
                selector.registrar_recompensa(mejora_absoluta)
            ratio = selector.seleccionar_ratio()
            mejor_costo_intervalo = mejor_costo

        num_iter += 1

    return ResultadoBenchmark(
        nombre="SA + UCB1",
        costo_final=mejor_costo,
        tiempo_total_s=time.perf_counter() - t_inicio,
        tiempo_entrenamiento_ml_s=selector.tiempo_entrenamiento_s,
        tiempo_inferencia_ml_s=selector.tiempo_inferencia_s,
        modelo_ml=selector,
    )

### Benchmark SA (movimientos, base) vs SA+UCB1

Compara **SA** por movimientos con ratio de enfriamiento fijo (celda
`sa-base-code`) contra **SA+UCB1**, la misma busqueda por movimientos
N1/N2/N3/N4 pero con el ratio de enfriamiento elegido por un bandido UCB1 en
vez de fijo (celda `04b06824`), sobre `wlp01`-`wlp04`, 5 semillas por
instancia.

Los dos algoritmos comparten exactamente el mismo tipo de paso (movimientos
individuales N1/N2/N3/N4 con Metropolis; la unica diferencia es como se elige
el ratio de enfriamiento), asi que un mismo `n_movimientos` les da
presupuestos comparables -- a diferencia del rediseno anterior como Busqueda
Local Iterada, cuyas iteraciones (perturbar + busqueda local completa)
costaban mucho mas que un movimiento y por eso necesitaban igualar el
presupuesto por tiempo de reloj en vez de por conteo. Por eso aqui se iguala
directamente por conteo: ambos reciben el mismo `n_movimientos`, sin medir ni
limitar por tiempo.

In [409]:
import os as _os_sab
import time as _tsab
import numpy as np

_EJECUTAR_BENCHMARK_SA_UCB1 = False

if _EJECUTAR_BENCHMARK_SA_UCB1:

    _INST_SA_BENCH      = ["wlp01.dzn", "wlp02.dzn", "wlp03.dzn", "wlp04.dzn"]
    _REF_SA_BENCH        = {"wlp01": 28716, "wlp02": 52952, "wlp03": 64296, "wlp04": 84633}
    _SEMILLAS_SA_BENCH   = [0, 1, 2, 3, 4]
    _N_MOV_SA_BENCH      = 500_000

    _sa_res = {
        nombre: {"SA": {"gaps": [], "t": []},
                 "SA+UCB1": {"gaps": [], "t": []}}
        for nombre in [a.replace(".dzn", "") for a in _INST_SA_BENCH]
    }

    for archivo in _INST_SA_BENCH:
        nombre = archivo.replace(".dzn", "")
        datos  = leer_instancia_dzn(_os_sab.path.join("Instaces_MS-CFLP-CI/Public", archivo))
        ref    = _REF_SA_BENCH[nombre]

        print(f"\n  {nombre}")
        for semilla in _SEMILLAS_SA_BENCH:
            t0 = _tsab.perf_counter()
            r_base = recocido_simulado(datos, n_movimientos=_N_MOV_SA_BENCH, semilla=semilla)
            t_base   = _tsab.perf_counter() - t0
            gap_base = 100 * (r_base.costo_final - ref) / ref
            _sa_res[nombre]["SA"]["gaps"].append(gap_base)
            _sa_res[nombre]["SA"]["t"].append(t_base)

            t0 = _tsab.perf_counter()
            r_ml = recocido_simulado_ml(
                datos, n_movimientos=_N_MOV_SA_BENCH, semilla=semilla,
            )
            t_ml   = _tsab.perf_counter() - t0
            gap_ml = 100 * (r_ml.costo_final - ref) / ref
            _sa_res[nombre]["SA+UCB1"]["gaps"].append(gap_ml)
            _sa_res[nombre]["SA+UCB1"]["t"].append(t_ml)

            print(f"    seed={semilla}  SA: {gap_base:+.2f}% {t_base:.1f}s  "
                  f"SA+UCB1: {gap_ml:+.2f}% {t_ml:.1f}s  Δgap={gap_ml-gap_base:+.2f}pp")

    instancias_sa = [a.replace(".dzn", "") for a in _INST_SA_BENCH]
    print(f"\n{'='*70}")
    print(f"  {'':<10}  {'── SA base ──':^18}  {'── SA+UCB1 ──':^18}  {'Δgap':>6}")
    print(f"  {'Inst.':<10}  {'gap%':>7}  {'σ':>5}  {'t(s)':>5}  "
          f"{'gap%':>7}  {'σ':>5}  {'t(s)':>5}  {'(pp)':>6}")
    print(f"  {'-'*70}")

    all_base, all_ml = [], []
    for nombre in instancias_sa:
        bg = np.mean(_sa_res[nombre]["SA"]["gaps"])
        bs = np.std(_sa_res[nombre]["SA"]["gaps"])
        bt = np.mean(_sa_res[nombre]["SA"]["t"])
        mg = np.mean(_sa_res[nombre]["SA+UCB1"]["gaps"])
        ms = np.std(_sa_res[nombre]["SA+UCB1"]["gaps"])
        mt = np.mean(_sa_res[nombre]["SA+UCB1"]["t"])
        all_base.append(bg); all_ml.append(mg)
        print(f"  {nombre:<10}  {bg:>+7.2f}  {bs:>5.2f}  {bt:>5.1f}  "
              f"{mg:>+7.2f}  {ms:>5.2f}  {mt:>5.1f}  {mg-bg:>+6.2f}")

    print(f"  {'-'*70}")
    mb = np.mean(all_base); mm = np.mean(all_ml)
    print(f"  {'Media':<10}  {mb:>+7.2f}  {'':<5}  {'':<5}  {mm:>+7.2f}  {'':<5}  {'':<5}  {mm-mb:>+6.2f}")
    print(f"  {'='*70}")
else:
    print("Benchmark SA vs SA+UCB1 omitido (_EJECUTAR_BENCHMARK_SA_UCB1 = False)")


Benchmark SA vs SA+UCB1 omitido (_EJECUTAR_BENCHMARK_SA_UCB1 = False)


## 5. ALNS (Adaptive Large Neighborhood Search) + seleccion adaptativa de operadores

ALNS (Ropke y Pisinger, 2006) construye cada iteracion en dos fases: **destruye**
una parte de la solucion actual -- elimina varios clientes de sus instalaciones
-- y la **repara** reinsertandolos con una heuristica constructiva.

`SelectorOperadorALNS` sustituye la eleccion uniforme de operador (destruccion y
reparacion) por una seleccion adaptativa: cada operador acumula un peso segun
el resultado de las iteraciones en que se uso, y ese peso determina la
probabilidad de elegirlo en las siguientes.

### Operadores de destruccion

Cada operador recibe un tamano `q` (clientes a eliminar) y libera esos clientes de
sus instalaciones actuales, cerrando cualquier instalacion que se quede sin
clientes:

- **D1 -- Aleatorio**: elimina `q` clientes al azar. El operador mas simple; sirve
  de referencia y de diversificacion pura.
- **D2 -- Peor coste**: elimina los `q` clientes cuyo coste de envio actual es mas
  alto. Ataca directamente las asignaciones mas caras de la solucion.
- **D3 -- Relacionado (Shaw removal)**: parte de un cliente semilla al azar y va
  anadiendo, uno a uno, el cliente restante mas "relacionado" con el ultimo
  elegido -- aquel cuyo coste cruzado de servirse mutuamente desde la instalacion
  del otro es menor --. Agrupa clientes que tendria sentido reasignar juntos.
- **D4 -- Cierre de instalacion**: cierra una instalacion abierta al azar y libera
  a todos sus clientes de golpe. Es la unica forma de que ALNS reconsidere
  directamente que instalaciones estan abiertas, mas que reordenar clientes entre
  las ya abiertas.
- **D5 -- Incompatibilidad**: elimina los `q` clientes con mas incompatibles
  actualmente asignados (con un pequeno termino aleatorio para desempatar).
  Especifico del MS-CFLP-CI: un cliente con muchos incompatibles ya colocados
  tiene su propia reinsercion fuertemente restringida, algo que D1-D4 no
  explotan porque ninguno razona sobre la estructura de incompatibilidad.

In [410]:
def _liberar_cliente_alns(cliente, asignada, restante, abierta, clientes_por_instalacion, demanda):
    origen = asignada[cliente]
    d = demanda[cliente]
    restante[origen] += d
    clientes_por_instalacion[origen].discard(cliente)
    asignada[cliente] = -1
    if not clientes_por_instalacion[origen]:
        abierta[origen] = False


def _destruir_aleatorio(asignada, restante, abierta, clientes_por_instalacion, demanda, q, num_clientes):
    candidatos = [j for j in range(num_clientes) if asignada[j] != -1]
    q = min(q, len(candidatos))
    eliminados = random.sample(candidatos, q)
    for j in eliminados:
        _liberar_cliente_alns(j, asignada, restante, abierta, clientes_por_instalacion, demanda)
    return eliminados


def _destruir_peor_coste(asignada, restante, abierta, clientes_por_instalacion, demanda, envio, q, num_clientes):
    candidatos = [j for j in range(num_clientes) if asignada[j] != -1]
    costes = [(envio[asignada[j]][j] * demanda[j], j) for j in candidatos]
    costes.sort(reverse=True)
    q = min(q, len(costes))
    eliminados = [j for _, j in costes[:q]]
    for j in eliminados:
        _liberar_cliente_alns(j, asignada, restante, abierta, clientes_por_instalacion, demanda)
    return eliminados


def _destruir_relacionado(asignada, restante, abierta, clientes_por_instalacion, demanda, envio, q, num_clientes):
    candidatos = [j for j in range(num_clientes) if asignada[j] != -1]
    q = min(q, len(candidatos))
    if q == 0:
        return []
    semilla = random.choice(candidatos)
    elegidos = [semilla]
    restantes = set(candidatos) - {semilla}
    while len(elegidos) < q and restantes:
        ref = elegidos[-1]
        f_ref = asignada[ref]

        def _relacion(k, f_ref=f_ref, ref=ref):
            return envio[f_ref][k] + envio[asignada[k]][ref]

        siguiente = min(restantes, key=_relacion)
        elegidos.append(siguiente)
        restantes.discard(siguiente)
    for j in elegidos:
        _liberar_cliente_alns(j, asignada, restante, abierta, clientes_por_instalacion, demanda)
    return elegidos


def _destruir_instalacion(asignada, restante, abierta, clientes_por_instalacion, demanda, num_instalaciones):
    abiertas = [i for i in range(num_instalaciones) if abierta[i]]
    if not abiertas:
        return []
    instalacion = random.choice(abiertas)
    eliminados = list(clientes_por_instalacion[instalacion])
    for j in eliminados:
        _liberar_cliente_alns(j, asignada, restante, abierta, clientes_por_instalacion, demanda)
    return eliminados


def _destruir_por_incompatibilidad(asignada, restante, abierta, clientes_por_instalacion,
                                    demanda, incompatibles, q, num_clientes):
    candidatos = [j for j in range(num_clientes) if asignada[j] != -1]
    q = min(q, len(candidatos))
    if q == 0:
        return []
    puntuadas = []
    for j in candidatos:
        conflicto = sum(1 for k in incompatibles[j] if asignada[k] != -1)
        puntuadas.append((conflicto + random.random(), j))
    puntuadas.sort(reverse=True)
    eliminados = [j for _, j in puntuadas[:q]]
    for j in eliminados:
        _liberar_cliente_alns(j, asignada, restante, abierta, clientes_por_instalacion, demanda)
    return eliminados


### Operadores de reparacion

Reciben la lista de clientes eliminados y los reinsertan uno a uno en instalaciones
factibles (con capacidad y sin incompatibilidades):

- **R1 -- Greedy**: procesa los clientes eliminados en orden aleatorio e inserta
  cada uno en su instalacion factible mas barata en ese momento.
- **R2 -- Arrepentimiento (regret-2)**: para cada cliente eliminado calcula la
  diferencia entre su mejor y su segunda mejor instalacion factible (su
  "arrepentimiento" si se pospone), e inserta primero a quien mas tiene que
  perder.

In [411]:
def _mejor_insercion_factible(cliente, asignada, restante, abierta, num_instalaciones, demanda, costo_fijo, envio, incompatibles):
    d = demanda[cliente]
    inc = incompatibles[cliente]
    prohibidas = {asignada[k] for k in inc if asignada[k] != -1}
    mejor_costo, mejor_inst = float("inf"), None
    for i in range(num_instalaciones):
        if restante[i] < d:
            continue
        if i in prohibidas:
            continue
        c = envio[i][cliente] * d
        if not abierta[i]:
            c += costo_fijo[i]
        if c < mejor_costo:
            mejor_costo, mejor_inst = c, i
    return mejor_costo, mejor_inst


def _asignar_cliente(cliente, instalacion, asignada, restante, abierta, clientes_por_instalacion, demanda):
    d = demanda[cliente]
    asignada[cliente] = instalacion
    restante[instalacion] -= d
    clientes_por_instalacion[instalacion].add(cliente)
    abierta[instalacion] = True


def _reparar_greedy(eliminados, asignada, restante, abierta, clientes_por_instalacion,
                     num_instalaciones, demanda, costo_fijo, envio, incompatibles):
    orden = list(eliminados)
    random.shuffle(orden)
    for j in orden:
        _, inst = _mejor_insercion_factible(
            j, asignada, restante, abierta, num_instalaciones, demanda, costo_fijo, envio, incompatibles
        )
        if inst is not None:
            _asignar_cliente(j, inst, asignada, restante, abierta, clientes_por_instalacion, demanda)


def _reparar_regret2(eliminados, asignada, restante, abierta, clientes_por_instalacion,
                      num_instalaciones, demanda, costo_fijo, envio, incompatibles):

    def _mejor_y_segunda(j):
        d = demanda[j]
        inc = incompatibles[j]
        prohibidas = {asignada[k] for k in inc if asignada[k] != -1}
        mejor_c, mejor_i = float("inf"), None
        segunda_c = float("inf")
        for i in range(num_instalaciones):
            if restante[i] < d or i in prohibidas:
                continue
            c = envio[i][j] * d + (0 if abierta[i] else costo_fijo[i])
            if c < mejor_c:
                segunda_c = mejor_c
                mejor_c, mejor_i = c, i
            elif c < segunda_c:
                segunda_c = c
        return mejor_c, mejor_i, segunda_c

    orden = []
    for j in eliminados:
        mejor_c, mejor_i, segunda_c = _mejor_y_segunda(j)
        regret = (segunda_c - mejor_c) if mejor_i is not None and segunda_c < float("inf") else 0.0
        orden.append((regret, j, mejor_i))
    orden.sort(key=lambda t: t[0], reverse=True)

    for _, j, mejor_i in orden:
        d = demanda[j]
        cabe = mejor_i is not None and restante[mejor_i] >= d and not any(
            asignada[k] == mejor_i for k in incompatibles[j]
        )
        if cabe:
            _asignar_cliente(j, mejor_i, asignada, restante, abierta, clientes_por_instalacion, demanda)
        else:
            _, inst = _mejor_insercion_factible(
                j, asignada, restante, abierta, num_instalaciones, demanda, costo_fijo, envio, incompatibles
            )
            if inst is not None:
                _asignar_cliente(j, inst, asignada, restante, abierta, clientes_por_instalacion, demanda)

### Algoritmo base: ALNS con seleccion uniforme de operadores

Cada iteracion parte de la solucion actual y aplica tres pasos: elegir un tamano de
destruccion `q` (entre el 5% y el 20% de los clientes), destruir `q` clientes con
uno de los cinco operadores anteriores, y repararlos con uno de los dos
operadores constructivos. La solucion reparada se acepta con el criterio de
Metropolis (mismo mecanismo que en el capitulo de SA, con calendario de
enfriamiento geometrico fijo -- aqui no hay componente ML en la temperatura, solo
en la eleccion de operador). El algoritmo base elige el operador de destruccion y
el de reparacion **uniformemente al azar** en cada iteracion; la variante con ML
(mas abajo) los elige segun pesos aprendidos.

In [412]:
def _aplicar_destroy(tipo, asignada, restante, abierta, clientes_por_instalacion,
                      demanda, envio, costo_fijo, q, num_clientes, num_instalaciones,
                      incompatibles=None):
    if tipo == 0:
        return _destruir_aleatorio(asignada, restante, abierta, clientes_por_instalacion, demanda, q, num_clientes)
    if tipo == 1:
        return _destruir_peor_coste(asignada, restante, abierta, clientes_por_instalacion, demanda, envio, q, num_clientes)
    if tipo == 2:
        return _destruir_relacionado(asignada, restante, abierta, clientes_por_instalacion, demanda, envio, q, num_clientes)
    if tipo == 3:
        return _destruir_instalacion(asignada, restante, abierta, clientes_por_instalacion, demanda, num_instalaciones)
    return _destruir_por_incompatibilidad(asignada, restante, abierta, clientes_por_instalacion,
                                          demanda, incompatibles, q, num_clientes)


def _aplicar_repair(tipo, eliminados, asignada, restante, abierta, clientes_por_instalacion,
                     num_instalaciones, demanda, costo_fijo, envio, incompatibles):
    if tipo == 0:
        _reparar_greedy(eliminados, asignada, restante, abierta, clientes_por_instalacion,
                         num_instalaciones, demanda, costo_fijo, envio, incompatibles)
    else:
        _reparar_regret2(eliminados, asignada, restante, abierta, clientes_por_instalacion,
                          num_instalaciones, demanda, costo_fijo, envio, incompatibles)


def alns_base(
    datos: DatosCFLP,
    n_iter: int = 2_000,
    ratio_enfriamiento: float = 0.995,
    semilla: int = 0,
    tiempo_limite_s: float | None = None,
    usar_d5: bool = True,
) -> ResultadoBenchmark:
    random.seed(semilla)
    t_inicio = time.perf_counter()
    num_instalaciones = datos["numero_instalaciones"]
    num_clientes      = datos["numero_clientes"]
    costo_fijo        = datos["costo_fijo_instalacion"]
    demanda           = datos["demanda_cliente"]
    envio             = datos["costo_envio"]
    incompatibles     = _precomputar_incompatibles(num_clientes, datos["pares_incompatibles"])

    abierta, asignada, restante = construir_solucion_greedy_aleatoria(
        num_instalaciones, num_clientes, datos["capacidad_instalacion"], costo_fijo, demanda, envio, 0.3,
        incompatibles,
    )
    abierta, asignada, restante = mejorar_solucion_con_busqueda_local(
        abierta, asignada, restante, num_instalaciones, num_clientes,
        datos["capacidad_instalacion"], costo_fijo, demanda, envio, incompatibles,
    )
    costo = calcular_costo_total(abierta, asignada, costo_fijo, demanda, envio)
    mejor_costo = costo

    clientes_por_instalacion = [set() for _ in range(num_instalaciones)]
    for j in range(num_clientes):
        clientes_por_instalacion[asignada[j]].add(j)

    q_min = max(3, num_clientes // 20)
    q_max = max(q_min + 1, num_clientes // 5)
    temperatura = costo * 0.02

    for _ in range(n_iter):
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break

        ab_prev, asig_prev, rest_prev = list(abierta), list(asignada), list(restante)
        cpi_prev = [set(s) for s in clientes_por_instalacion]

        tipo_destroy = random.randint(0, 4 if usar_d5 else 3)
        tipo_repair  = random.randint(0, 1)
        q = random.randint(q_min, q_max)

        eliminados = _aplicar_destroy(
            tipo_destroy, asignada, restante, abierta, clientes_por_instalacion,
            demanda, envio, costo_fijo, q, num_clientes, num_instalaciones,
            incompatibles,
        )
        _aplicar_repair(
            tipo_repair, eliminados, asignada, restante, abierta, clientes_por_instalacion,
            num_instalaciones, demanda, costo_fijo, envio, incompatibles,
        )

        costo_nuevo = calcular_costo_total(abierta, asignada, costo_fijo, demanda, envio)
        delta = costo_nuevo - costo

        acepta = delta <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta / temperatura))
        if acepta:
            costo = costo_nuevo
            if costo < mejor_costo:
                mejor_costo = costo
        else:
            abierta, asignada, restante = ab_prev, asig_prev, rest_prev
            clientes_por_instalacion = cpi_prev

        temperatura *= ratio_enfriamiento
        if temperatura < 1e-10:
            temperatura = mejor_costo * 5e-4

    return ResultadoBenchmark(
        nombre="ALNS",
        costo_final=mejor_costo,
        tiempo_total_s=time.perf_counter() - t_inicio,
    )

### Componente ML: seleccion adaptativa de operadores

`SelectorOperadorALNS` mantiene un peso por operador (5 de destruccion, 2 de
reparacion) y elige cada uno por ruleta: la probabilidad de elegir un operador es
proporcional a su peso frente al resto. Cada `tamano_segmento` iteraciones (un
"segmento") se acumula, por operador usado, una puntuacion segun el resultado de
esa iteracion:

- **33 puntos** si produjo un nuevo mejor global.
- **20 puntos** si mejoro la solucion actual (sin llegar a mejor global).
- **9 puntos** si se acepto aunque no mejorara.
- **0 puntos** si se rechazo.

Al final del segmento, el peso de cada operador se actualiza como una media
exponencial entre su peso anterior y su rendimiento medio en el segmento
(puntuacion acumulada entre veces usado), y los contadores se reinician para el
siguiente segmento. Es el mismo esquema de pesos adaptativos por ruleta que
introdujeron Ropke y Pisinger (2006) para ALNS.

In [413]:
class SelectorOperadorALNS:

    SIGMA_MEJOR_GLOBAL = 33.0
    SIGMA_MEJORA       = 20.0

    def __init__(self, n_destroy: int = 4, n_repair: int = 2,
                 tamano_segmento: int = 100, factor_reaccion: float = 0.2) -> None:
        self.tamano_segmento = tamano_segmento
        self.factor_reaccion = factor_reaccion
        self.pesos_destroy   = [1.0] * n_destroy
        self.pesos_repair    = [1.0] * n_repair
        self._score_destroy  = [0.0] * n_destroy
        self._usos_destroy   = [0] * n_destroy
        self._score_repair   = [0.0] * n_repair
        self._usos_repair    = [0] * n_repair
        self._iter_segmento  = 0
        self.tiempo_entrenamiento_s: float = 0.0
        self.tiempo_inferencia_s: float    = 0.0

    def _elegir(self, pesos: list[float]) -> int:
        t0 = time.perf_counter()
        total = sum(pesos)
        r = random.random() * total
        acumulado = 0.0
        elegido = len(pesos) - 1
        for i, w in enumerate(pesos):
            acumulado += w
            if r <= acumulado:
                elegido = i
                break
        self.tiempo_inferencia_s += time.perf_counter() - t0
        return elegido

    def elegir_destroy(self) -> int:
        return self._elegir(self.pesos_destroy)

    def elegir_repair(self) -> int:
        return self._elegir(self.pesos_repair)

    def registrar(self, tipo_destroy: int, tipo_repair: int, es_mejor_global: bool,
                  mejora_actual: bool) -> None:
        if es_mejor_global:
            sigma = self.SIGMA_MEJOR_GLOBAL
        elif mejora_actual:
            sigma = self.SIGMA_MEJORA
        else:
            sigma = 0.0

        self._score_destroy[tipo_destroy] += sigma
        self._usos_destroy[tipo_destroy]  += 1
        self._score_repair[tipo_repair]   += sigma
        self._usos_repair[tipo_repair]    += 1
        self._iter_segmento += 1

        if self._iter_segmento >= self.tamano_segmento:
            t0 = time.perf_counter()
            r = self.factor_reaccion
            for i in range(len(self.pesos_destroy)):
                if self._usos_destroy[i] > 0:
                    rendimiento = self._score_destroy[i] / self._usos_destroy[i]
                    self.pesos_destroy[i] = (1 - r) * self.pesos_destroy[i] + r * rendimiento
            for i in range(len(self.pesos_repair)):
                if self._usos_repair[i] > 0:
                    rendimiento = self._score_repair[i] / self._usos_repair[i]
                    self.pesos_repair[i] = (1 - r) * self.pesos_repair[i] + r * rendimiento
            self._score_destroy = [0.0] * len(self.pesos_destroy)
            self._usos_destroy  = [0] * len(self.pesos_destroy)
            self._score_repair  = [0.0] * len(self.pesos_repair)
            self._usos_repair   = [0] * len(self.pesos_repair)
            self._iter_segmento = 0
            suma_d = sum(self.pesos_destroy)
            if suma_d > 0:
                factor_d = len(self.pesos_destroy) / suma_d
                self.pesos_destroy = [p * factor_d for p in self.pesos_destroy]
            suma_r = sum(self.pesos_repair)
            if suma_r > 0:
                factor_r = len(self.pesos_repair) / suma_r
                self.pesos_repair = [p * factor_r for p in self.pesos_repair]
            self.tiempo_entrenamiento_s += time.perf_counter() - t0

### ALNS con selección adaptativa

Idéntico al algoritmo base salvo en un punto: en vez de sortear el operador de
destrucción y el de reparación de forma uniforme, se consulta a
`SelectorOperadorALNS` (selección por ruleta según los pesos vigentes) y se le
registra el resultado de cada iteración para que ajuste esos pesos por segmentos.

In [414]:
def alns_ml(
    datos: DatosCFLP,
    n_iter: int = 2_000,
    ratio_enfriamiento: float = 0.995,
    tamano_segmento: int | None = None,
    factor_reaccion: float = 0.2,
    semilla: int = 0,
    tiempo_limite_s: float | None = None,
    usar_d5: bool = True,
) -> ResultadoBenchmark:
    if tamano_segmento is None:
        if tiempo_limite_s is not None:
            tamano_segmento = max(20, round(750 * (115 / datos["numero_clientes"]) ** 0.7))
        else:
            tamano_segmento = max(20, n_iter // 100)
    random.seed(semilla)
    t_inicio = time.perf_counter()
    num_instalaciones = datos["numero_instalaciones"]
    num_clientes      = datos["numero_clientes"]
    costo_fijo        = datos["costo_fijo_instalacion"]
    demanda           = datos["demanda_cliente"]
    envio             = datos["costo_envio"]
    incompatibles     = _precomputar_incompatibles(num_clientes, datos["pares_incompatibles"])

    abierta, asignada, restante = construir_solucion_greedy_aleatoria(
        num_instalaciones, num_clientes, datos["capacidad_instalacion"], costo_fijo, demanda, envio, 0.3,
        incompatibles,
    )
    abierta, asignada, restante = mejorar_solucion_con_busqueda_local(
        abierta, asignada, restante, num_instalaciones, num_clientes,
        datos["capacidad_instalacion"], costo_fijo, demanda, envio, incompatibles,
    )
    costo = calcular_costo_total(abierta, asignada, costo_fijo, demanda, envio)
    mejor_costo = costo

    clientes_por_instalacion = [set() for _ in range(num_instalaciones)]
    for j in range(num_clientes):
        clientes_por_instalacion[asignada[j]].add(j)

    q_min = max(3, num_clientes // 20)
    q_max = max(q_min + 1, num_clientes // 5)
    temperatura = costo * 0.02
    selector = SelectorOperadorALNS(n_destroy=(5 if usar_d5 else 4), tamano_segmento=tamano_segmento, factor_reaccion=factor_reaccion)

    for _ in range(n_iter):
        if tiempo_limite_s is not None and time.perf_counter() - t_inicio >= tiempo_limite_s:
            break

        ab_prev, asig_prev, rest_prev = list(abierta), list(asignada), list(restante)
        cpi_prev = [set(s) for s in clientes_por_instalacion]

        tipo_destroy = selector.elegir_destroy()
        tipo_repair  = selector.elegir_repair()
        q = random.randint(q_min, q_max)

        eliminados = _aplicar_destroy(
            tipo_destroy, asignada, restante, abierta, clientes_por_instalacion,
            demanda, envio, costo_fijo, q, num_clientes, num_instalaciones,
            incompatibles,
        )
        _aplicar_repair(
            tipo_repair, eliminados, asignada, restante, abierta, clientes_por_instalacion,
            num_instalaciones, demanda, costo_fijo, envio, incompatibles,
        )

        costo_nuevo = calcular_costo_total(abierta, asignada, costo_fijo, demanda, envio)
        delta = costo_nuevo - costo

        acepta = delta <= 0 or (temperatura > 1e-10 and random.random() < math.exp(-delta / temperatura))
        es_mejor_global = False
        if acepta:
            mejora_actual = costo_nuevo < costo
            costo = costo_nuevo
            if costo < mejor_costo:
                mejor_costo = costo
                es_mejor_global = True
        else:
            mejora_actual = False
            abierta, asignada, restante = ab_prev, asig_prev, rest_prev
            clientes_por_instalacion = cpi_prev

        selector.registrar(tipo_destroy, tipo_repair, es_mejor_global, mejora_actual)

        temperatura *= ratio_enfriamiento
        if temperatura < 1e-10:
            temperatura = mejor_costo * 5e-4

    return ResultadoBenchmark(
        nombre="ALNS + Adaptativo",
        costo_final=mejor_costo,
        tiempo_total_s=time.perf_counter() - t_inicio,
        tiempo_entrenamiento_ml_s=selector.tiempo_entrenamiento_s,
        tiempo_inferencia_ml_s=selector.tiempo_inferencia_s,
        modelo_ml=selector,
    )

### Benchmark ALNS (selección uniforme, base) vs ALNS+adaptativo

Compara **ALNS** con selección uniforme de operador contra **ALNS+adaptativo**
(pesos por segmentos vía `SelectorOperadorALNS`), sobre `wlp01`-`wlp04`, 5 semillas
por instancia, `n_iter=10000` para ambos (comparten el mismo tipo de iteración
--destruir + reparar--, así que un mismo presupuesto de iteraciones da comparaciones
comparables, igual que en TS).

In [415]:
import os as _os_alns
import time as _talns
import numpy as np

_EJECUTAR_BENCHMARK_ALNS_ADAPT = False

if _EJECUTAR_BENCHMARK_ALNS_ADAPT:

    _INST_ALNS_BENCH    = ["wlp01.dzn", "wlp02.dzn", "wlp03.dzn", "wlp04.dzn"]
    _REF_ALNS_BENCH      = {"wlp01": 28716, "wlp02": 52952, "wlp03": 64296, "wlp04": 84633}
    _SEMILLAS_ALNS_BENCH = [0, 1, 2, 3, 4]
    _N_ITER_ALNS_BENCH   = 10_000

    _alns_res = {
        nombre: {"ALNS": {"gaps": [], "t": []},
                 "ALNS+Adaptativo": {"gaps": [], "t": []}}
        for nombre in [a.replace(".dzn", "") for a in _INST_ALNS_BENCH]
    }

    for archivo in _INST_ALNS_BENCH:
        nombre = archivo.replace(".dzn", "")
        datos  = leer_instancia_dzn(_os_alns.path.join("Instaces_MS-CFLP-CI/Public", archivo))
        ref    = _REF_ALNS_BENCH[nombre]

        print(f"\n  {nombre}")
        for semilla in _SEMILLAS_ALNS_BENCH:
            t0 = _talns.perf_counter()
            r_base = alns_base(datos, n_iter=_N_ITER_ALNS_BENCH, semilla=semilla)
            t_base   = _talns.perf_counter() - t0
            gap_base = 100 * (r_base.costo_final - ref) / ref
            _alns_res[nombre]["ALNS"]["gaps"].append(gap_base)
            _alns_res[nombre]["ALNS"]["t"].append(t_base)

            t0 = _talns.perf_counter()
            r_ml = alns_ml(datos, n_iter=_N_ITER_ALNS_BENCH, semilla=semilla)
            t_ml   = _talns.perf_counter() - t0
            gap_ml = 100 * (r_ml.costo_final - ref) / ref
            _alns_res[nombre]["ALNS+Adaptativo"]["gaps"].append(gap_ml)
            _alns_res[nombre]["ALNS+Adaptativo"]["t"].append(t_ml)

            print(f"    seed={semilla}  ALNS: {gap_base:+.2f}% {t_base:.1f}s  "
                  f"ALNS+Adapt: {gap_ml:+.2f}% {t_ml:.1f}s  Δgap={gap_ml-gap_base:+.2f}pp")

    instancias_alns = [a.replace(".dzn", "") for a in _INST_ALNS_BENCH]
    print(f"\n{'='*74}")
    print(f"  {'':<10}  {'── ALNS base ──':^20}  {'── ALNS+Adapt ──':^20}  {'Δgap':>6}")
    print(f"  {'Inst.':<10}  {'gap%':>7}  {'σ':>5}  {'t(s)':>5}  "
          f"{'gap%':>7}  {'σ':>5}  {'t(s)':>5}  {'(pp)':>6}")
    print(f"  {'-'*74}")

    all_base, all_ml = [], []
    for nombre in instancias_alns:
        bg = np.mean(_alns_res[nombre]["ALNS"]["gaps"])
        bs = np.std(_alns_res[nombre]["ALNS"]["gaps"])
        bt = np.mean(_alns_res[nombre]["ALNS"]["t"])
        mg = np.mean(_alns_res[nombre]["ALNS+Adaptativo"]["gaps"])
        ms = np.std(_alns_res[nombre]["ALNS+Adaptativo"]["gaps"])
        mt = np.mean(_alns_res[nombre]["ALNS+Adaptativo"]["t"])
        all_base.append(bg); all_ml.append(mg)
        print(f"  {nombre:<10}  {bg:>+7.2f}  {bs:>5.2f}  {bt:>5.1f}  "
              f"{mg:>+7.2f}  {ms:>5.2f}  {mt:>5.1f}  {mg-bg:>+6.2f}")

    print(f"  {'-'*74}")
    mb = np.mean(all_base); mm = np.mean(all_ml)
    print(f"  {'Media':<10}  {mb:>+7.2f}  {'':<5}  {'':<5}  {mm:>+7.2f}  {'':<5}  {'':<5}  {mm-mb:>+6.2f}")
    print(f"  {'='*74}")
else:
    print("Benchmark ALNS vs ALNS+Adaptativo omitido (_EJECUTAR_BENCHMARK_ALNS_ADAPT = False)")

Benchmark ALNS vs ALNS+Adaptativo omitido (_EJECUTAR_BENCHMARK_ALNS_ADAPT = False)


## 6. Benchmark sobre instancias MS-CFLP-CI

Ejecuta los ocho algoritmos (4 pares base/ML) sobre las instancias `wlp01`-`wlp08`
de `Instaces_MS-CFLP-CI/Public/`, con **5 semillas** por instancia, y compara los
costes obtenidos contra la referencia de CPLEX/Gurobi (optimo probado en
`wlp01`-`wlp03`, mejor valor conocido en el resto, marcado con `*`).

**Presupuesto de tiempo compartido entre familias.** Las cuatro familias (y
cada par base/+ML) comparten el **mismo presupuesto de tiempo de reloj**
(`_PRESUPUESTO_BASE_S = 6.0` segundos para `wlp01` en el benchmark final). Ese
presupuesto se escala con el numero de clientes de cada instancia respecto a
`wlp01` (`_C_REFERENCIA_PRESUPUESTO = 115`) siguiendo una potencia n^1.3. GVNS
puede parar antes de agotar su presupuesto al converger a un optimo local
respecto a los cuatro vecindarios -- es su criterio de parada estandar --,
mientras que GRASP, SA y ALNS agotan el tiempo asignado.

Las instancias tienen tamanos crecientes: desde 50 almacenes / 115 tiendas (`wlp01`)
hasta 500 / 1277 (`wlp08`). El benchmark esta controlado por el flag
`_EJECUTAR_BENCHMARK_FINAL` de la celda siguiente. Al final de la seccion se
muestra, ademas de las tablas de resultados, un resumen de lo que aprendio
cada componente de ML sobre cada instancia.

In [416]:
_C_REFERENCIA_PRESUPUESTO = 115

def benchmarcar_instancia(
    datos: DatosCFLP, grado_poly: int = 2, semilla: int = 0, presupuesto_s: float = 15.0,
) -> dict[str, ResultadoBenchmark]:
    resultados: dict[str, ResultadoBenchmark] = {}

    F, C  = datos["numero_instalaciones"], datos["numero_clientes"]
    cap   = datos["capacidad_instalacion"]
    cf    = datos["costo_fijo_instalacion"]
    dem   = datos["demanda_cliente"]
    envio = datos["costo_envio"]
    pares_inc = datos["pares_incompatibles"]

    _trabajo = lambda n: n ** 1.3
    presupuesto = presupuesto_s * (_trabajo(C) / _trabajo(_C_REFERENCIA_PRESUPUESTO))

    t0 = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()):
        r_grasp = ejecutar_grasp(
            F, C, cap, cf, dem, envio,
            alfa=0.5, numero_maximo_de_iteraciones=1_000_000, semilla_aleatoria=semilla,
            usar_alfa_adaptativo=False, pares_incompatibles=pares_inc,
            tiempo_limite_s=presupuesto,
        )
    resultados["GRASP"] = ResultadoBenchmark(
        nombre="GRASP",
        costo_final=r_grasp["costo_total_minimo"],
        tiempo_total_s=time.perf_counter() - t0,
    )

    resultados["GRASP+ML"] = ejecutar_grasp_ml(
        datos, n_iter=1_000_000, semilla=semilla,
        verbose=False, tiempo_limite_s=presupuesto, usar_thompson=True,
    )

    incompatibles = _precomputar_incompatibles(C, pares_inc)
    ab0, asig0, rest0 = seleccionar_solucion_aleatoria_del_pool(datos, tamano_pool=5, semilla_aleatoria=semilla)
    ab0, asig0, rest0 = mejorar_solucion_con_busqueda_local(
        ab0, asig0, rest0, F, C, cap, cf, dem, envio, incompatibles,
    )

    t0 = time.perf_counter()
    costo_gvns = _ejecutar_gvns_iter_vnd(
        datos, ab0, asig0, rest0, k_maximo=3, n_iter=100_000, semilla=semilla,
        tiempo_limite_s=presupuesto,
    )
    resultados["GVNS"] = ResultadoBenchmark(
        nombre="GVNS", costo_final=costo_gvns, tiempo_total_s=time.perf_counter() - t0,
    )

    t0 = time.perf_counter()
    costo_gvns_ql, agente_ql, agente_k_ql = _ejecutar_gvns_iter_ql(
        datos, ab0, asig0, rest0, k_maximo=3, n_iter=100_000, semilla=semilla,
        tiempo_limite_s=presupuesto,
    )
    resultados["GVNS+QL"] = ResultadoBenchmark(
        nombre="GVNS+QL", costo_final=costo_gvns_ql, tiempo_total_s=time.perf_counter() - t0,
        modelo_ml=AgentesGVNSQL(agente_ql, agente_k_ql),
    )

    resultados["SA"] = recocido_simulado(
        datos, n_movimientos=100_000_000, semilla=semilla, tiempo_limite_s=presupuesto,
    )
    resultados["SA+ML"] = recocido_simulado_ml(
        datos, n_movimientos=100_000_000, semilla=semilla, tiempo_limite_s=presupuesto,
    )

    resultados["ALNS"] = alns_base(
        datos, n_iter=10_000_000, semilla=semilla, tiempo_limite_s=presupuesto,
    )
    resultados["ALNS+Adaptativo"] = alns_ml(
        datos, n_iter=10_000_000, semilla=semilla, tiempo_limite_s=presupuesto,
    )

    return resultados

In [417]:
_CPLEX = {
    "wlp01": 28716, "wlp02": 52952, "wlp03": 64296,
    "wlp04": None,  "wlp05": None,  "wlp06": None,
    "wlp07": None,  "wlp08": None,
}
_GUROBI = {
    "wlp01": 28716, "wlp02": 52952, "wlp03": 64296,
    "wlp04": 84633, "wlp05": 103857, "wlp06": 111654,
    "wlp07": 162277, "wlp08": 187938,
}

REFERENCIA = {
    inst: min(v for v in [_CPLEX[inst], _GUROBI[inst]] if v is not None)
    for inst in _CPLEX
}

def _tag(inst):
    c, g = _CPLEX[inst], _GUROBI[inst]
    if c is not None and g is not None and c == g:
        return "✓"
    return "*"


In [418]:
import pickle

_DIR_MODELOS = "resultados_benchmark"
_ARCHIVO_MODELOS_TXT = os.path.join(_DIR_MODELOS, "modelos_aprendidos.txt")
_ARCHIVO_MODELOS_PKL = os.path.join(_DIR_MODELOS, "modelos_aprendidos.pkl")


def _formatear_resumen_modelos(nombre: str, modelos: dict) -> str:
    lineas = [f"\n{'='*70}", f"  {nombre}", f"{'='*70}"]

    if "GRASP+ML" in modelos:
        selector = modelos["GRASP+ML"]
        if isinstance(selector, SelectorAlfaThompson):
            lineas.append(f"  GRASP+ML  (Thompson Sampling, {selector.total_tiradas} tiradas)")
            for alfa, a, b in zip(selector.ALFAS, selector.a, selector.b):
                media = a / (a + b)
                lineas.append(f"    alfa={alfa:<4} Beta(a={a:.2f}, b={b:.2f})  media={media:.4f}")
        elif isinstance(selector, SelectorAlfaUCB):
            lineas.append(f"  GRASP+ML  (bandido UCB1, {selector.total_tiradas} tiradas)")
            for alfa, veces, suma in zip(selector.ALFAS, selector.conteo, selector.suma_recompensa):
                media = suma / veces if veces > 0 else float("nan")
                lineas.append(f"    alfa={alfa:<4} veces_elegido={veces:<4} recompensa_media={media:.4f}")
        else:
            alfa_aprendido = selector.seleccionar_alfa()
            lineas.append(f"  GRASP+ML  (regresión polinómica grado {selector.grado}, "
                           f"{len(selector.historial_alfa)} muestras acumuladas)")
            lineas.append(f"    alfa* recomendado por el modelo final: {alfa_aprendido:.3f}")

    if "GVNS+QL" in modelos:
        lineas.append(f"  GVNS+QL")
        lineas.append(f"    {modelos['GVNS+QL'].resumen()}")

    if "SA+ML" in modelos:
        selector = modelos["SA+ML"]
        lineas.append(f"  SA+UCB1  (estadísticas finales por brazo)")
        for ratio, veces, suma in zip(selector.RATIOS, selector.conteo, selector.suma_recompensa):
            media = suma / veces if veces > 0 else float("nan")
            lineas.append(f"    ratio={ratio:<6} veces_elegido={veces:<4} recompensa_media={media:.4f}")

    if "ALNS+Adaptativo" in modelos:
        selector = modelos["ALNS+Adaptativo"]
        lineas.append(f"  ALNS+Adaptativo  (pesos finales)")
        nombres_destroy = ["D1 Aleatorio", "D2 Peor coste", "D3 Relacionado", "D4 Cierre instalación"]
        nombres_repair  = ["R1 Greedy", "R2 Regret-2"]
        for n, w in zip(nombres_destroy, selector.pesos_destroy):
            lineas.append(f"    {n:<24} peso={w:.2f}")
        for n, w in zip(nombres_repair, selector.pesos_repair):
            lineas.append(f"    {n:<24} peso={w:.2f}")

    return "\n".join(lineas)


def _guardar_modelos_incremental(nombre: str, modelos: dict, modelos_ml_completo: dict) -> None:
    os.makedirs(_DIR_MODELOS, exist_ok=True)
    resumen_txt = _formatear_resumen_modelos(nombre, modelos)
    with open(_ARCHIVO_MODELOS_TXT, "a") as f:
        f.write(resumen_txt + "\n")
    with open(_ARCHIVO_MODELOS_PKL, "wb") as f:
        pickle.dump(modelos_ml_completo, f)



In [419]:
_EJECUTAR_BENCHMARK_FINAL = True

_PRESUPUESTO_BASE_S = 6.0
_SEMILLAS_BENCHMARK_FINAL = list(range(5))
_ALGORITMOS_BENCHMARK = ["GRASP", "GRASP+ML", "GVNS", "GVNS+QL", "SA", "SA+ML", "ALNS", "ALNS+Adaptativo"]
_PARES_BENCHMARK = [("GRASP", "GRASP+ML"), ("GVNS", "GVNS+QL"), ("SA", "SA+ML"), ("ALNS", "ALNS+Adaptativo")]

INSTANCIAS_BENCHMARK = [
    "wlp01.dzn", "wlp02.dzn", "wlp03.dzn", "wlp04.dzn",
    "wlp05.dzn", "wlp06.dzn", "wlp07.dzn", "wlp08.dzn",
]
data_dir = "Instaces_MS-CFLP-CI/Public"

resultados_benchmark: dict[str, dict] = {}
modelos_ml: dict[str, dict[str, object]] = {}

if _EJECUTAR_BENCHMARK_FINAL:
    for archivo in INSTANCIAS_BENCHMARK:
        nombre = archivo.replace(".dzn", "")
        datos  = leer_instancia_dzn(os.path.join(data_dir, archivo))
        F, C   = datos["numero_instalaciones"], datos["numero_clientes"]
        ref    = REFERENCIA[nombre]

        gaps    = {alg: [] for alg in _ALGORITMOS_BENCHMARK}
        tiempos = {alg: [] for alg in _ALGORITMOS_BENCHMARK}

        print(f"\n{nombre} {_tag(nombre)}  (F={F}, C={C})")
        for semilla in _SEMILLAS_BENCHMARK_FINAL:
            res = benchmarcar_instancia(
                datos, grado_poly=_MEJOR_GRADO, semilla=semilla, presupuesto_s=_PRESUPUESTO_BASE_S,
            )
            for alg in _ALGORITMOS_BENCHMARK:
                gaps[alg].append(100 * (res[alg].costo_final - ref) / ref)
                tiempos[alg].append(res[alg].tiempo_total_s)
            if semilla == 0:
                modelos_ml[nombre] = {
                    alg: res[alg].modelo_ml for alg in _ALGORITMOS_BENCHMARK
                    if res[alg].modelo_ml is not None
                }
                _guardar_modelos_incremental(nombre, modelos_ml[nombre], modelos_ml)
            print(f"  seed={semilla}  " + "  ".join(
                f"{alg}={gaps[alg][-1]:+.1f}%/{tiempos[alg][-1]:.1f}s" for alg in _ALGORITMOS_BENCHMARK
            ), flush=True)

        resultados_benchmark[nombre] = {"F": F, "C": C, "gaps": gaps, "tiempos": tiempos}
else:
    print("Benchmark final omitido (_EJECUTAR_BENCHMARK_FINAL = False)")



wlp01 ✓  (F=50, C=115)
  seed=0  GRASP=+5.8%/6.0s  GRASP+ML=+4.8%/6.0s  GVNS=+5.7%/6.0s  GVNS+QL=+3.8%/6.0s  SA=+4.6%/6.0s  SA+ML=+5.8%/6.0s  ALNS=+7.0%/6.0s  ALNS+Adaptativo=+5.9%/6.0s
  seed=1  GRASP=+6.1%/6.0s  GRASP+ML=+5.1%/6.0s  GVNS=+4.2%/6.0s  GVNS+QL=+4.0%/6.0s  SA=+5.2%/6.0s  SA+ML=+7.7%/6.0s  ALNS=+5.3%/6.0s  ALNS+Adaptativo=+5.9%/6.0s
  seed=2  GRASP=+5.2%/6.0s  GRASP+ML=+6.0%/6.0s  GVNS=+5.2%/6.0s  GVNS+QL=+4.7%/6.0s  SA=+6.9%/6.0s  SA+ML=+7.4%/6.0s  ALNS=+5.8%/6.0s  ALNS+Adaptativo=+5.6%/6.0s
  seed=3  GRASP=+5.5%/6.0s  GRASP+ML=+5.8%/6.0s  GVNS=+3.8%/6.0s  GVNS+QL=+4.2%/6.0s  SA=+6.2%/6.0s  SA+ML=+5.2%/6.0s  ALNS=+8.9%/6.0s  ALNS+Adaptativo=+8.3%/6.0s
  seed=4  GRASP=+5.2%/6.0s  GRASP+ML=+4.4%/6.0s  GVNS=+4.4%/6.0s  GVNS+QL=+3.8%/6.0s  SA=+5.5%/6.0s  SA+ML=+5.7%/6.0s  ALNS=+5.6%/6.0s  ALNS+Adaptativo=+4.5%/6.0s

wlp02 ✓  (F=100, C=253)
  seed=0  GRASP=+7.4%/16.8s  GRASP+ML=+8.0%/16.8s  GVNS=+5.6%/16.7s  GVNS+QL=+6.0%/16.7s  SA=+9.2%/16.7s  SA+ML=+8.9%/16.7s  ALNS=+7.1%/

In [422]:
if resultados_benchmark:
    col_w = 9

    print("Gap medio respecto a la referencia (%, media ± σ sobre 5 semillas):")
    header = f"{'Instancia':<10}"
    for alg in _ALGORITMOS_BENCHMARK:
        header += f"  {alg:>13}"
    print(header)
    print("-" * len(header))

    medias_por_alg = {alg: [] for alg in _ALGORITMOS_BENCHMARK}
    for nombre, datos_r in resultados_benchmark.items():
        line = f"{nombre+' '+_tag(nombre):<10}"
        for alg in _ALGORITMOS_BENCHMARK:
            g = datos_r["gaps"][alg]
            media, sigma = np.mean(g), np.std(g)
            medias_por_alg[alg].append(media)
            line += f"  {media:>6.2f}±{sigma:<5.2f}"
        print(line)
    print("-" * len(header))
    line = f"{'Media':<10}"
    for alg in _ALGORITMOS_BENCHMARK:
        line += f"  {np.mean(medias_por_alg[alg]):>13.2f}"
    print(line)

    print()
    print("Mejora del algoritmo ML sobre su base (Δgap en pp, media sobre 5 semillas; + = ML mejora):")
    header2 = f"{'Instancia':<10}"
    for base, ml in _PARES_BENCHMARK:
        header2 += f"  {base+' vs '+ml:>18}"
    print(header2)
    print("-" * len(header2))

    deltas_por_par = {(base, ml): [] for base, ml in _PARES_BENCHMARK}
    for nombre, datos_r in resultados_benchmark.items():
        line = f"{nombre+' '+_tag(nombre):<10}"
        for base, ml in _PARES_BENCHMARK:
            delta = np.mean(datos_r["gaps"][base]) - np.mean(datos_r["gaps"][ml])
            deltas_por_par[(base, ml)].append(delta)
            line += f"  {delta:>15.2f} pp"
        print(line)
    print("-" * len(header2))
    line = f"{'Media':<10}"
    for base, ml in _PARES_BENCHMARK:
        line += f"  {np.mean(deltas_por_par[(base, ml)]):>15.2f} pp"
    print(line)
else:
    print("No hay resultados. Activa _EJECUTAR_BENCHMARK_FINAL y ejecuta la celda anterior.")


Gap medio respecto a la referencia (%, media ± σ sobre 5 semillas):
Instancia           GRASP       GRASP+ML           GVNS        GVNS+QL             SA          SA+ML           ALNS  ALNS+Adaptativo
------------------------------------------------------------------------------------------------------------------------------------
wlp01 ✓       5.56±0.35     5.22±0.61     4.67±0.69     4.11±0.31     5.67±0.79     6.36±0.99     6.51±1.34     6.05±1.23 
wlp02 ✓       7.46±0.49     7.06±0.82     5.98±0.23     5.59±0.21     9.05±0.57     7.91±0.63     8.52±1.30     7.30±1.14 
wlp03 ✓       9.50±0.31     9.00±0.30     6.25±0.76     6.29±0.54    12.12±1.08     9.73±0.84     9.52±1.17     8.87±0.89 
wlp04 *       9.59±0.18     9.10±0.21     6.85±0.73     6.16±0.60    11.28±0.85    10.97±0.49     9.96±0.56     8.69±1.37 
wlp05 *      10.93±0.54     9.98±0.57     8.28±0.42     7.91±0.22    11.98±0.47    11.68±0.21    11.44±0.60     9.95±0.45 
wlp06 *      11.16±0.56    10.71±0.30     9.16±0.62

### Que aprendieron los modelos de ML

Para cada instancia se guarda el modelo/agente entrenado en la semilla 0 (una
de las 5 usadas en el benchmark), a modo de ejemplo interpretable de lo que
cada componente de ML aprendio sobre esa instancia concreta -- no es un
promedio sobre las 5 semillas, solo una muestra representativa de una
ejecucion real.

In [423]:
if not modelos_ml and os.path.exists(_ARCHIVO_MODELOS_PKL):
    with open(_ARCHIVO_MODELOS_PKL, "rb") as f:
        modelos_ml = pickle.load(f)
    print(f"(modelos_ml recargado desde {_ARCHIVO_MODELOS_PKL})")

if modelos_ml:
    for nombre, modelos in modelos_ml.items():
        print(_formatear_resumen_modelos(nombre, modelos))
else:
    print("No hay modelos guardados. Ejecuta primero el benchmark (_EJECUTAR_BENCHMARK_FINAL = True).")



  wlp01
  GRASP+ML  (Thompson Sampling, 207 tiradas)
    alfa=0.1  Beta(a=17.79, b=8.21)  media=0.6842
    alfa=0.3  Beta(a=71.23, b=13.77)  media=0.8380
    alfa=0.5  Beta(a=13.58, b=5.42)  media=0.7150
    alfa=0.7  Beta(a=28.28, b=7.72)  media=0.7856
    alfa=0.9  Beta(a=41.03, b=9.97)  media=0.8046
  GVNS+QL
    ε=0.050 | pasos=4611 | t_ref=0.9ms | Q(arr)=[0.014,0.018,0.029,0.011]->N3 | Q(N1✗)=[0.004,0.000,0.000,0.001]->N1 | Q(N1✓)=[0.004,0.003,0.001,0.001]->N1 | Q(N2✗)=[0.002,0.003,0.000,0.001]->N2 | Q(N2✓)=[0.002,0.003,0.001,0.001]->N2 | Q(N3✗)=[0.002,0.000,0.002,0.003]->N4 | Q(N3✓)=[0.008,0.014,0.002,0.005]->N2 | Q(N4✗)=[0.001,0.002,0.001,0.001]->N2 | Q(N4✓)=[0.002,0.001,0.001,0.001]->N1 || k-UCB1: k=1:0.0000(236) | k=2:0.0001(236) | k=3:0.0002(236)
  SA+UCB1  (estadísticas finales por brazo)
    ratio=0.9    veces_elegido=17   recompensa_media=0.0741
    ratio=0.95   veces_elegido=17   recompensa_media=0.0757
    ratio=0.99   veces_elegido=16   recompensa_media=0.0332
    rati